# Tarvo · exploração inicial dos dados (set/2026)

**Objetivo:** olhar os arquivos *como chegam*, sem limpar nada, e listar os casos que a solução vai ter que tratar.

**Como ler este notebook**
- Tudo é lido como **texto** (`dtype=str`). Nenhuma coluna é corrigida na origem.
- Quando preciso de número, crio uma coluna auxiliar com sufixo `_f` (só para diagnóstico).
- Nada aqui é a solução final: são observações para virar **hipóteses** e **perguntas ao financeiro**.
- Rodar com este notebook na mesma pasta dos arquivos do data pack.

In [1]:
import re, io, zipfile, email, hashlib, unicodedata
from email import policy
from pathlib import Path
import xml.etree.ElementTree as ET
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.max_rows", 120)

DATA_DIR = Path("input")   # pasta com os arquivos de entrada
F = {
    "orbe":      DATA_DIR / "01_orbe_relatorio_comissoes_set2026.csv",
    "taxas":     DATA_DIR / "02_tabela_taxas_contratadas.xlsx",
    "ponte":     DATA_DIR / "03_email_ponte_cambio_relatorio_set2026.eml",
    "plano":     DATA_DIR / "04_plano_de_contas_fenix_erp.csv",
    "aurora":    DATA_DIR / "05a_extrato_banco_aurora_cc4471_set2026.csv",
    "meridiano": DATA_DIR / "05b_extrato_banco_meridiano_cc90233_set2026.ofx",
}
faltando = [p.name for p in F.values() if not p.exists()]
assert not faltando, f"Arquivos não encontrados em {DATA_DIR.resolve()}: {faltando}"


# ---- helpers de leitura (não corrigem dados; só abrem os arquivos) ----
def read_text(path):
    """Lê bytes e tenta UTF-8; se falhar, cai para cp1252. Devolve (texto, encoding)."""
    raw = Path(path).read_bytes()
    for enc in ("utf-8", "cp1252"):
        try:
            return raw.decode(enc), enc
        except UnicodeDecodeError:
            pass
    return raw.decode("latin-1"), "latin-1"


def read_csv_raw(path, **kw):
    txt, enc = read_text(path)
    df = pd.read_csv(io.StringIO(txt), sep=";", dtype=str, keep_default_na=False, **kw)
    return df, enc


def br_num(s):
    """SÓ PARA DIAGNÓSTICO: '1.234,56' -> 1234.56 ; '2,69' -> 2.69 ; '0.0149' -> 0.0149."""
    if s is None or (not isinstance(s, str) and pd.isna(s)):
        return float("nan")
    s = str(s).strip().replace("R$", "").replace("%", "").strip()
    if s == "":
        return float("nan")
    if "," in s:
        s = s.replace(".", "").replace(",", ".")
    try:
        return float(s)
    except ValueError:
        return float("nan")


def digitos(s):
    return re.sub(r"\D", "", str(s))


def cnpj14(s):
    return digitos(s).zfill(14)


def norm(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    return re.sub(r"[^A-Z0-9 ]", " ", s.upper()).strip()


def prefixo(desc, n=2):
    """Primeiras n palavras da descrição, só para agrupar lançamentos parecidos."""
    palavras = re.findall(r"[A-Za-zÀ-ÿ]+", str(desc))
    return " ".join(palavras[:n]).upper()


def read_xlsx_raw(src):
    """Lê .xlsx sem openpyxl. Devolve ({aba: DataFrame(linha Excel x coluna Excel)}, [(ref, texto)])."""
    zf = zipfile.ZipFile(io.BytesIO(src) if isinstance(src, (bytes, bytearray)) else src)
    M = "{http://schemas.openxmlformats.org/spreadsheetml/2006/main}"
    R = "{http://schemas.openxmlformats.org/officeDocument/2006/relationships}"
    ss = []
    if "xl/sharedStrings.xml" in zf.namelist():
        for si in ET.fromstring(zf.read("xl/sharedStrings.xml")).findall(M + "si"):
            ss.append("".join(t.text or "" for t in si.iter(M + "t")))
    rels = {r.get("Id"): r.get("Target") for r in ET.fromstring(zf.read("xl/_rels/workbook.xml.rels"))}
    wb = ET.fromstring(zf.read("xl/workbook.xml"))

    def col_idx(c):
        n = 0
        for ch in c:
            n = n * 26 + ord(ch) - 64
        return n

    grids = {}
    for sh in wb.find(M + "sheets"):
        alvo = rels[sh.get(R + "id")]
        caminho = alvo.lstrip("/") if alvo.startswith("/") else "xl/" + alvo
        root = ET.fromstring(zf.read(caminho))
        cells = {}
        for c in root.iter(M + "c"):
            t, v = c.get("t"), c.find(M + "v")
            if t == "inlineStr":
                val = "".join(x.text or "" for x in c.iter(M + "t"))
            elif v is None:
                continue
            elif t == "s":
                val = ss[int(v.text)]
            else:
                val = v.text
            m = re.match(r"([A-Z]+)(\d+)", c.get("r"))
            cells[(int(m.group(2)), m.group(1))] = val
        if not cells:
            grids[sh.get("name")] = pd.DataFrame()
            continue
        rows = sorted({r for r, _ in cells})
        cols = sorted({c for _, c in cells}, key=col_idx)
        g = pd.DataFrame(index=rows, columns=cols, dtype=object)
        for (r, c), val in cells.items():
            g.loc[r, c] = val
        grids[sh.get("name")] = g
    comentarios = []
    for n in zf.namelist():
        if n.startswith("xl/comments"):
            for cm in ET.fromstring(zf.read(n)).iter(M + "comment"):
                comentarios.append((cm.get("ref"), "".join(t.text or "" for t in cm.iter(M + "t"))))
    return grids, comentarios

print("ok · pasta:", DATA_DIR.resolve())

ok · pasta: C:\Users\pedro\OneDrive\Área de Trabalho\ITA_projects\Hackaton


## 1. Inventário dos arquivos
Confere se os arquivos soltos são iguais aos que estão dentro do `.zip` (hash MD5).

In [2]:
ZIP = DATA_DIR / "Tarvo_Data_Pack_set2026.zip"
zf = zipfile.ZipFile(ZIP) if ZIP.exists() else None
linhas = []
for k, p in F.items():
    r = {"fonte": k, "arquivo": p.name, "tamanho_kb": round(p.stat().st_size / 1024, 1)}
    if zf is not None:
        r["igual_ao_zip"] = p.name in zf.namelist() and hashlib.md5(p.read_bytes()).hexdigest() == hashlib.md5(zf.read(p.name)).hexdigest()
    linhas.append(r)
pd.DataFrame(linhas)

,fonte,arquivo,tamanho_kb,igual_ao_zip
0,orbe,01_orbe_relatorio_comissoes_set2026.csv,216.3,True
1,taxas,02_tabela_taxas_contratadas.xlsx,9.9,True
2,ponte,03_email_ponte_cambio_relatorio_set2026.eml,41.6,False
3,plano,04_plano_de_contas_fenix_erp.csv,3.1,True
4,aurora,05a_extrato_banco_aurora_cc4471_set2026.csv,6.8,False
5,meridiano,05b_extrato_banco_meridiano_cc90233_set2026.ofx,17.9,True


## 2. Orbe — relatório de comissões (checkout)
Uma linha por transação: vendas, estornos, antecipações e reversões.

In [3]:
orbe, enc_orbe = read_csv_raw(F["orbe"])
print("encoding:", enc_orbe, "| linhas:", len(orbe), "| colunas:", len(orbe.columns))
display(orbe.head(5))
print("\nCélulas vazias por coluna (texto vazio):")
display((orbe == "").sum().to_frame("vazios").T)

encoding: utf-8 | linhas: 1943 | colunas: 14


,id_transacao,data_movimento,cod_estabelecimento,nome_estabelecimento,tipo_lancamento,modalidade,qtd_parcelas,bandeira,valor_bruto,taxa_aplicada_pct,valor_taxa,valor_liquido,id_referencia,data_venda_original
0,TX88410932,01/09/2026,1018325654,CASA DO ESTUDANTE VIAGENS,VENDA,CREDITO A VISTA,1,MASTERCARD,"608,88","2,69","16,37","592,51",,
1,TX88410975,01/09/2026,1018325654,CASA DO ESTUDANTE VIAGENS,VENDA,CREDITO A VISTA,1,MASTERCARD,"614,86","2,69","16,53","598,33",,
2,TX88410833,01/09/2026,1098729012,BEM-VINDO EXCHANGE,VENDA,CREDITO A VISTA,1,MASTERCARD,"834,61","2,69","22,45","812,16",,
3,TX88410128,01/09/2026,1018325654,CASA DO ESTUDANTE VIAGENS,VENDA,CREDITO PARCELADO,11,ELO,"5773,57","3,79","218,81","5554,76",,
4,TX88410316,01/09/2026,1000960410,OXFORD PREP CURSOS,VENDA,CREDITO A VISTA,1,VISA,"1645,47","2,59","42,61","1602,86",,



Células vazias por coluna (texto vazio):


,id_transacao,data_movimento,cod_estabelecimento,nome_estabelecimento,tipo_lancamento,modalidade,qtd_parcelas,bandeira,valor_bruto,taxa_aplicada_pct,valor_taxa,valor_liquido,id_referencia,data_venda_original
vazios,0,0,0,0,0,44,44,227,0,0,0,0,1878,1878


In [4]:
print("Tipos de lançamento:")
display(orbe.tipo_lancamento.value_counts().to_frame("linhas"))
print("Modalidades (inclui vazio nas antecipações):")
display(orbe.modalidade.value_counts().to_frame("linhas"))
print("Bandeiras:")
display(orbe.bandeira.value_counts().to_frame("linhas"))
print("Prefixos de id_transacao:")
display(orbe.id_transacao.str[:2].value_counts().to_frame("linhas"))
print("id_transacao duplicado:", orbe.id_transacao.duplicated().sum())
datas = pd.to_datetime(orbe.data_movimento, format="%d/%m/%Y")
print("Período de data_movimento:", datas.min().date(), "→", datas.max().date(), "| dias distintos:", datas.nunique())

Tipos de lançamento:


,linhas
tipo_lancamento,
VENDA,1840
ESTORNO,59
ANTECIPACAO,38
REVERSAO ANTECIPACAO,6


Modalidades (inclui vazio nas antecipações):


,linhas
modalidade,
CREDITO PARCELADO,951
CREDITO A VISTA,591
PIX,183
DEBITO,174
,44


Bandeiras:


,linhas
bandeira,
MASTERCARD,356
ELO,343
AMEX,343
VISA,342
HIPERCARD,332
,227


Prefixos de id_transacao:


,linhas
id_transacao,
TX,1899
AN,44


id_transacao duplicado: 0
Período de data_movimento: 2026-09-01 → 2026-09-30 | dias distintos: 30


### 2.1 Nomes inconsistentes por estabelecimento
O **código do estabelecimento (EC)** é estável; o **nome** não. Casar por nome seria erro.

In [5]:
nomes = (orbe.groupby("cod_estabelecimento")
         .agg(linhas=("id_transacao", "size"),
              n_nomes=("nome_estabelecimento", "nunique"),
              nomes=("nome_estabelecimento", lambda s: " | ".join(sorted(s.unique()))))
         .sort_values("n_nomes", ascending=False))
print("ECs distintos:", len(nomes), "| ECs com mais de um nome:", (nomes.n_nomes > 1).sum())
display(nomes)
por_nome = orbe.groupby("nome_estabelecimento").cod_estabelecimento.nunique()
print("Nomes que aparecem em mais de um EC:", por_nome[por_nome > 1].to_dict())

ECs distintos: 22 | ECs com mais de um nome: 8


,linhas,n_nomes,nomes
cod_estabelecimento,,,
1037647039,168,3,MUNDO ABERTO INTERC LTDA | MUNDO ABERTO INTERCAMBIO | Mundo Aberto
1018325654,119,2,CASA DO ESTUDANTE VIAG | CASA DO ESTUDANTE VIAGENS
1020660828,100,2,EDU ABROAD CONSULTORIA | EDUABROAD
1023622151,76,2,LISBOA CURSOS E VIAGENS | LISBOA CURSOS VIAGENS
1061760196,80,2,CANGURU INTERCAMBIO | CANGURU INTERCAMBIOS
1041180753,105,2,MAPLE STUDY | MAPLE STUDY ASSESSORIA
1098729012,76,2,BEM VINDO EXCHANGE | BEM-VINDO EXCHANGE
1075624862,107,2,BRIDGE IDIOMAS | ESCOLA BRIDGE IDIOMAS
1000960410,98,1,OXFORD PREP CURSOS


Nomes que aparecem em mais de um EC: {}


### 2.2 Estornos
Cada estorno aponta (`id_referencia`) para a venda original. A venda original está no arquivo de setembro?

In [6]:
orig = orbe.drop_duplicates("id_transacao").set_index("id_transacao")
est = orbe[orbe.tipo_lancamento == "ESTORNO"].copy()
est["origem_no_arquivo"] = est.id_referencia.isin(orig.index)
est["bruto_origem"] = est.id_referencia.map(lambda r: orig.loc[r, "valor_bruto"] if r in orig.index else "")
est["venda_original_dt"] = pd.to_datetime(est.data_venda_original, format="%d/%m/%Y")
est["venda_de_mes_anterior"] = est.venda_original_dt < pd.Timestamp("2026-09-01")
print("Estornos:", len(est), "| com a venda original no arquivo:", est.origem_no_arquivo.sum(),
      "| de mês anterior:", est.venda_de_mes_anterior.sum())
est[["id_transacao", "data_movimento", "cod_estabelecimento", "nome_estabelecimento", "modalidade", "qtd_parcelas",
     "valor_bruto", "valor_taxa", "id_referencia", "data_venda_original", "origem_no_arquivo", "bruto_origem"]]

Estornos: 59 | com a venda original no arquivo: 50 | de mês anterior: 9


,id_transacao,data_movimento,cod_estabelecimento,nome_estabelecimento,modalidade,qtd_parcelas,valor_bruto,valor_taxa,id_referencia,data_venda_original,origem_no_arquivo,bruto_origem
37,TX88444848,01/09/2026,1000960410,OXFORD PREP CURSOS,CREDITO PARCELADO,10,"-2193,16","-83,12",TX87651902,22/08/2026,False,
135,TX88444937,02/09/2026,1070531381,CAMPUS GLOBAL,CREDITO A VISTA,1,"-1532,07","-41,21",TX88410045,01/09/2026,True,"1532,07"
156,TX88444966,03/09/2026,1014537966,DUBLIN HUB INTERCAMBIO,CREDITO PARCELADO,10,"-3115,05","-118,06",TX87440146,11/08/2026,False,
165,TX88444985,03/09/2026,1037647039,Mundo Aberto,CREDITO A VISTA,1,"-1773,30","-44,15",TX87131975,05/08/2026,False,
210,TX88445015,04/09/2026,1064304917,SOL NASCENTE VIAGENS,CREDITO PARCELADO,4,"-3531,43","-112,65",TX87162998,10/08/2026,False,
247,TX88445053,04/09/2026,1043396662,HORIZONTE VIAGENS TURISMO,CREDITO PARCELADO,4,"-2094,44","-66,81",TX87805689,19/08/2026,False,
260,TX88445084,05/09/2026,1070531381,CAMPUS GLOBAL,CREDITO PARCELADO,4,"-8917,21","-266,62",TX87551657,29/08/2026,False,
431,TX88445133,07/09/2026,1018325654,CASA DO ESTUDANTE VIAG,DEBITO,1,"-369,39","-5,50",TX87983388,21/08/2026,False,
521,TX88445191,09/09/2026,1066100982,FLUENTE IDIOMAS ONLINE,DEBITO,1,"-956,03","-14,24",TX88415134,05/09/2026,True,"956,03"
525,TX88445205,09/09/2026,1074504011,ROTA GLOBAL INTERCAMBIO,CREDITO A VISTA,1,"-642,65","-16,00",TX88415540,05/09/2026,True,"642,65"


### 2.3 Antecipações e reversões

In [7]:
ant = orbe[orbe.tipo_lancamento.str.contains("ANTECIPACAO")].copy()
print("Antecipações / reversões:", ant.tipo_lancamento.value_counts().to_dict())
print("Taxa aplicada nas antecipações:", ant.taxa_aplicada_pct.value_counts().to_dict())
rev = ant[ant.tipo_lancamento.str.startswith("REVERSAO")]
print("Reversões cuja antecipação original está no arquivo:", rev.id_referencia.isin(orbe.id_transacao).sum(), "de", len(rev))
ant[["id_transacao", "data_movimento", "cod_estabelecimento", "nome_estabelecimento", "tipo_lancamento",
     "valor_bruto", "taxa_aplicada_pct", "valor_taxa", "valor_liquido", "id_referencia", "data_venda_original"]]

Antecipações / reversões: {'ANTECIPACAO': 38, 'REVERSAO ANTECIPACAO': 6}


Taxa aplicada nas antecipações: {'2,10': 44}
Reversões cuja antecipação original está no arquivo: 6 de 6


,id_transacao,data_movimento,cod_estabelecimento,nome_estabelecimento,tipo_lancamento,valor_bruto,taxa_aplicada_pct,valor_taxa,valor_liquido,id_referencia,data_venda_original
58,AN88444882,01/09/2026,1061760196,CANGURU INTERCAMBIO,ANTECIPACAO,"31765,12","2,10","667,06","31098,06",,
122,AN88444915,02/09/2026,1037647039,MUNDO ABERTO INTERCAMBIO,ANTECIPACAO,"48042,38","2,10","1008,88","47033,50",,
169,AN88444988,03/09/2026,1041180753,MAPLE STUDY ASSESSORIA,ANTECIPACAO,"28066,62","2,10","589,39","27477,23",,
244,AN88445034,04/09/2026,1014537966,DUBLIN HUB INTERCAMBIO,ANTECIPACAO,"47173,28","2,10","990,63","46182,65",,
250,AN88445076,04/09/2026,1038867654,KIWI EXPERIENCE TURISMO,ANTECIPACAO,"52314,21","2,10","1098,59","51215,62",,
264,AN88445093,05/09/2026,1038867654,KIWI EXPERIENCE TURISMO,REVERSAO ANTECIPACAO,"-52314,21","2,10","-1098,59","-51215,62",AN88445076,04/09/2026
278,AN88445114,05/09/2026,1037647039,Mundo Aberto,REVERSAO ANTECIPACAO,"-48042,38","2,10","-1008,88","-47033,50",AN88444915,02/09/2026
479,AN88445136,08/09/2026,1000960410,OXFORD PREP CURSOS,ANTECIPACAO,"40021,58","2,10","840,45","39181,13",,
506,AN88445141,09/09/2026,1074504011,ROTA GLOBAL INTERCAMBIO,ANTECIPACAO,"11884,08","2,10","249,56","11634,52",,
507,AN88445172,09/09/2026,1018325654,CASA DO ESTUDANTE VIAG,ANTECIPACAO,"4937,92","2,10","103,69","4834,23",,


### 2.4 A conta fecha em cada linha?
Confere, linha a linha: `valor_taxa ≈ bruto × taxa%` e `bruto − taxa = líquido` (tolerância de R$ 0,01).

In [8]:
d = orbe.copy()
for c in ["valor_bruto", "taxa_aplicada_pct", "valor_taxa", "valor_liquido"]:
    d[c + "_f"] = d[c].map(br_num)
d["taxa_calc"] = (d.valor_bruto_f * d.taxa_aplicada_pct_f / 100).round(2)
d["dif_taxa"] = (d.valor_taxa_f - d.taxa_calc).round(2)
d["dif_liq"] = (d.valor_bruto_f - d.valor_taxa_f - d.valor_liquido_f).round(2)
ruins = d[(d.dif_taxa.abs() > 0.01) | (d.dif_liq.abs() > 0.01)]
print("Linhas que não fecham:", len(ruins), "de", len(d))
print("Distribuição de dif_taxa:", d.dif_taxa.value_counts().head(6).to_dict())
ruins[["id_transacao", "tipo_lancamento", "valor_bruto", "taxa_aplicada_pct", "valor_taxa", "valor_liquido", "taxa_calc", "dif_taxa", "dif_liq"]]

Linhas que não fecham: 0 de 1943
Distribuição de dif_taxa: {-0.01: 969, 0.0: 942, 0.01: 32}


,id_transacao,tipo_lancamento,valor_bruto,taxa_aplicada_pct,valor_taxa,valor_liquido,taxa_calc,dif_taxa,dif_liq


A diferença de ±R$ 0,01 acima sugere que a Orbe **não arredonda do mesmo jeito que o pandas**. Testando três regras de arredondamento com `Decimal`:

In [9]:
from decimal import Decimal, ROUND_HALF_UP, ROUND_HALF_EVEN, ROUND_DOWN

def dec(s):
    return Decimal(s.replace(".", "").replace(",", "."))

q = Decimal("0.01")
regras = {"ROUND_HALF_UP (arredonda)": ROUND_HALF_UP,
          "ROUND_HALF_EVEN (banqueiro)": ROUND_HALF_EVEN,
          "ROUND_DOWN (trunca em direção a zero)": ROUND_DOWN}
acertos = {k: 0 for k in regras}
for _, r in orbe.iterrows():
    taxa_exata = dec(r.valor_bruto) * dec(r.taxa_aplicada_pct) / Decimal(100)
    for k, modo in regras.items():
        acertos[k] += taxa_exata.quantize(q, modo) == dec(r.valor_taxa)
pd.Series(acertos, name=f"linhas em que a regra reproduz valor_taxa (de {len(orbe)})").to_frame()

,linhas em que a regra reproduz valor_taxa (de 1943)
ROUND_HALF_UP (arredonda),942
ROUND_HALF_EVEN (banqueiro),942
ROUND_DOWN (trunca em direção a zero),1943


In [10]:
tot = d.groupby("tipo_lancamento")[["valor_bruto_f", "valor_taxa_f", "valor_liquido_f"]].agg(["count", "sum"]).round(2)
tot

valor_bruto_f             valor_taxa_f            valor_liquido_f            
                             count         sum        count        sum           count         sum
tipo_lancamento                                                                                   
ANTECIPACAO                     38  1451981.71           38   30491.41              38  1421490.30
ESTORNO                         59  -216346.03           59   -7108.69              59  -209237.34
REVERSAO ANTECIPACAO             6  -288606.34            6   -6060.69               6  -282545.65
VENDA                         1840  6549199.22         1840  214558.33            1840  6334640.89

### 2.5 Que taxa a Orbe aplicou, por estabelecimento e faixa?
Faixas do contrato: débito, crédito à vista, crédito 2–6x, crédito 7–12x, Pix e antecipação. Se um EC aparece com **mais de uma taxa na mesma faixa**, ou há erro de cobrança ou há vigência diferente.

In [11]:
def faixa(r):
    if "ANTECIPACAO" in r.tipo_lancamento:
        return "antecipacao"
    m = r.modalidade
    q = int(r.qtd_parcelas) if r.qtd_parcelas.strip().isdigit() else 0
    if m == "DEBITO":
        return "debito"
    if m == "PIX":
        return "pix"
    if m == "CREDITO A VISTA":
        return "credito_a_vista"
    if m == "CREDITO PARCELADO":
        return "credito_2_6x" if q <= 6 else "credito_7_12x"
    return "outro"

d["faixa"] = d.apply(faixa, axis=1)
print(d.faixa.value_counts().to_dict())
obs = (d[d.tipo_lancamento == "VENDA"]
       .groupby(["cod_estabelecimento", "faixa"])
       .agg(linhas=("id_transacao", "size"), n_taxas=("taxa_aplicada_pct_f", "nunique"),
            taxas=("taxa_aplicada_pct_f", lambda s: sorted(s.unique()))))
print("Pares (EC, faixa):", len(obs), "| com mais de uma taxa aplicada:", (obs.n_taxas > 1).sum())
obs[obs.n_taxas > 1]

{'credito_2_6x': 616, 'credito_a_vista': 591, 'credito_7_12x': 335, 'pix': 183, 'debito': 174, 'antecipacao': 44}
Pares (EC, faixa): 110 | com mais de uma taxa aplicada: 4


linhas  n_taxas         taxas
cod_estabelecimento faixa                                         
1074504011          credito_2_6x         41        2  [2.99, 3.49]
                    credito_7_12x        27        2  [3.59, 4.09]
                    credito_a_vista      55        2  [2.49, 2.99]
1075624862          credito_a_vista      22        2  [2.19, 2.59]

## 3. Tabela de taxas contratadas (planilha mantida à mão)

In [12]:
grids_tx, coment_tx = read_xlsx_raw(F["taxas"])
print("Abas:", list(grids_tx))
print("\nComentários de célula na planilha:")
for ref, txt in coment_tx:
    print(f"  {ref}: {txt}")
g = grids_tx["Taxas vigentes"]
print("\nCabeçalho / título (linhas 1 a 5):")
display(g.loc[1:5].dropna(how="all", axis=1))

Abas: ['Taxas vigentes', 'ANTIGO 2025 - nao usar']

Comentários de célula na planilha:
  A13: conversar com Diego (comercial) sobre condição especial

Cabeçalho / título (linhas 1 a 5):


,A,B,C,D,E,F,G,H,I,J,K,L
1,TARVO - TABELA DE TAXAS CONTRATADAS (atualizar sempre que o comercial fechar!!),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,última atualização: 02/09 - Bia,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,CARTÃO / PIX (MDR),NaN,NaN,NaN,NaN,NaN,REMESSA,NaN,NaN
5,Lojista,CNPJ,EC Orbe,Débito,Crédito à vista,Créd. 2-6x,Créd. 7-12x,Pix,Antecipação,Spread remessa,Vigência,Obs


In [13]:
ult = g.index[g["A"].astype(str).str.startswith("*")]
fim = (ult.min() - 1) if len(ult) else g.index.max()
tx = g.loc[6:fim].copy()
nomes_cols = ["lojista", "cnpj", "ec", "debito", "cred_vista", "cred_2_6x", "cred_7_12x", "pix", "antecipacao",
              "spread_remessa", "vigencia", "obs"]
print("Cabeçalho original (linha 5):", list(g.loc[5]))
tx = tx.reindex(columns=list(g.columns))
if tx.shape[1] == len(nomes_cols):
    tx.columns = nomes_cols
tx = tx[tx["lojista"].notna()].reset_index(drop=True)
print("Linhas de lojista:", len(tx))
print("Rodapé:", [g.loc[i, "A"] for i in ult])
tx

Cabeçalho original (linha 5): ['Lojista', 'CNPJ', 'EC Orbe', 'Débito', 'Crédito à vista', 'Créd. 2-6x', 'Créd. 7-12x', 'Pix', 'Antecipação', 'Spread remessa', 'Vigência', 'Obs']
Linhas de lojista: 25
Rodapé: ['* spread em branco = padrão']


,lojista,cnpj,ec,debito,cred_vista,cred_2_6x,cred_7_12x,pix,antecipacao,spread_remessa,vigencia,obs
0,Escola Bridge de Idiomas Ltda,22.224.579/0001-09,1075624862,0.0149,0.0259,0.0319,0.0379,0.0089,0.021,NaN,até 14/09/2026,renegociado - ver linha de baixo
1,Bridge Idiomas (NOVA),22.224.579/0001-09,1075624862,0.0149,0.0219,0.0319,0.0379,0.0089,0.021,NaN,a partir de 15/09/2026,só mudou o crédito à vista
2,Horizonte Viagens,78576234000179,NaN,0.0149,0.0269,0.0319,0.0379,0.0089,0.021,NaN,01/01/2026,
3,Kiwi Experience,07752116000126,NaN,0.0149,0.0259,0.0319,0.0379,0.0089,0.021,NaN,01/01/2026,EC pendente
4,Rota Global Intercâmbio Ltda,95.927.868/0001-48,1074504011,0.0149,0.0249,0.0299,0.0359,0.0089,0.021,0.015,01/01/2026,
5,Vela,20987545000140,1086291869,0.0149,0.0269,0.0319,0.0379,0.0089,0.021,NaN,01/01/2026,
6,Andes Expedições Ltda,85694108000184,1058961395,0.0139,0.0269,0.0319,0.0379,0.0089,0.021,NaN,01/01/2026,
7,Casa do Estudante Viagens Ltda,00520815000100,1018325654,0.0149,0.0269,0.0319,0.0379,0.0089,0.021,0.015,01/01/2026,cliente estratégico - ver c/ comercial
8,Fluente Idiomas Online Ltda,3229837000103,1066100982,0.0149,0.0269,0.0319,0.0379,0.007900000000000001,0.021,NaN,01/01/2026,
9,Mundo Aberto,17.217.888/0001-11,1037647039,0.0149,0.0249,0.0299,0.0379,0.0089,0.021,0.012,01/01/2026,spread negociado em jul/26


### 3.1 Qualidade da planilha: formato dos percentuais, CNPJ e EC

In [14]:
def fmt(v):
    if pd.isna(v):
        return "vazio"
    s = str(v)
    if "%" in s:
        return "texto_%"
    if re.fullmatch(r"\d+(\.\d+)?", s):
        return "numero"
    return "outro"

cols_taxa = ["debito", "cred_vista", "cred_2_6x", "cred_7_12x", "pix", "antecipacao", "spread_remessa"]
formatos = tx[cols_taxa].apply(lambda c: c.map(fmt))
print("Formato dos percentuais por coluna:")
display(formatos.apply(lambda c: c.value_counts()).fillna(0).astype(int))
print("Lojistas com percentual escrito como texto ('1,49%'):", tx.loc[(formatos == "texto_%").any(axis=1), "lojista"].tolist())

tx["cnpj_digitos"] = tx.cnpj.map(lambda s: digitos(s) if pd.notna(s) else "")
tx["cnpj_formatado"] = tx.cnpj.map(lambda s: bool(re.search(r"\D", str(s))) if pd.notna(s) else False)
print("\nCNPJ com pontuação:", int(tx.cnpj_formatado.sum()), "| só dígitos:", int((~tx.cnpj_formatado).sum()))
print("CNPJs com menos de 14 dígitos (zero à esquerda perdido):")
display(tx.loc[tx.cnpj_digitos.str.len() != 14, ["lojista", "cnpj", "cnpj_digitos"]])
print("Lojistas sem EC cadastrado:", tx.loc[tx.ec.isna(), "lojista"].tolist())
print("Spread de remessa em branco ('padrão'):", tx.loc[tx.spread_remessa.isna(), "lojista"].tolist())
print("\nObservações e vigências:")
display(tx.loc[tx.obs.notna() | tx.vigencia.astype(str).str.contains("até|partir"), ["lojista", "vigencia", "obs"]])
print("CNPJs repetidos na tabela:", tx.cnpj_digitos[tx.cnpj_digitos.duplicated(keep=False)].unique().tolist())

Formato dos percentuais por coluna:


,debito,cred_vista,cred_2_6x,cred_7_12x,pix,antecipacao,spread_remessa
numero,21,21,21,21,21,21,11
outro,0,0,0,0,0,0,1
texto_%,2,2,2,2,2,2,0
vazio,2,2,2,2,2,2,13


Lojistas com percentual escrito como texto ('1,49%'): ['Oxford Prep Cursos Ltda', 'Sol Nascente Viagens Ltda']

CNPJ com pontuação: 8 | só dígitos: 17
CNPJs com menos de 14 dígitos (zero à esquerda perdido):


,lojista,cnpj,cnpj_digitos
8,Fluente Idiomas Online Ltda,3229837000103,3229837000103
19,Campus Global Ltda,8780784000120,8780784000120


Lojistas sem EC cadastrado: ['Horizonte Viagens', 'Kiwi Experience', 'Ponte Aérea Estudantil Ltda', 'Nórdica Educação Internacional Ltda', 'Novo Horizonte Intercambio']
Spread de remessa em branco ('padrão'): ['Escola Bridge de Idiomas Ltda', 'Bridge Idiomas (NOVA)', 'Horizonte Viagens', 'Kiwi Experience', 'Vela', 'Andes Expedições Ltda', 'Fluente Idiomas Online Ltda', 'Ponte Aérea Estudantil Ltda', 'Oxford Prep Cursos Ltda', 'Sol Nascente Viagens Ltda', 'Trilha Jovem Turismo Ltda', 'Campus Global Ltda', 'Globo Trotter Viagens Ltda']

Observações e vigências:


,lojista,vigencia,obs
0,Escola Bridge de Idiomas Ltda,até 14/09/2026,renegociado - ver linha de baixo
1,Bridge Idiomas (NOVA),a partir de 15/09/2026,só mudou o crédito à vista
2,Horizonte Viagens,01/01/2026,
3,Kiwi Experience,01/01/2026,EC pendente
4,Rota Global Intercâmbio Ltda,01/01/2026,
5,Vela,01/01/2026,
6,Andes Expedições Ltda,01/01/2026,
7,Casa do Estudante Viagens Ltda,01/01/2026,cliente estratégico - ver c/ comercial
8,Fluente Idiomas Online Ltda,01/01/2026,
9,Mundo Aberto,01/01/2026,spread negociado em jul/26


CNPJs repetidos na tabela: ['22224579000109']


### 3.2 Orbe × tabela: os ECs conversam?

In [15]:
ecs_orbe = set(orbe.cod_estabelecimento)
ecs_tx = set(tx.ec.dropna().astype(str))
print("ECs na Orbe:", len(ecs_orbe), "| ECs na tabela:", len(ecs_tx))
sem_cad = (d[d.cod_estabelecimento.isin(ecs_orbe - ecs_tx)]
           .groupby("cod_estabelecimento")
           .agg(linhas=("id_transacao", "size"), bruto=("valor_bruto_f", "sum"), taxa=("valor_taxa_f", "sum"),
                nomes=("nome_estabelecimento", lambda s: " | ".join(sorted(s.unique())))))
print("\nECs que aparecem na Orbe mas NÃO estão na tabela:")
display(sem_cad.round(2))
print("Lojistas da tabela sem nenhuma linha na Orbe (EC ausente ou lojista só de remessa):")
display(tx.loc[~tx.ec.astype(str).isin(ecs_orbe), ["lojista", "ec", "obs"]])

ECs na Orbe: 22 | ECs na tabela: 19

ECs que aparecem na Orbe mas NÃO estão na tabela:


,linhas,bruto,taxa,nomes
cod_estabelecimento,,,,
1038867654,85,328428.75,10711.06,KIWI EXPERIENCE TURISMO
1043300981,73,246356.75,7767.49,HORIZONTE INTERCAMBIO
1043396662,75,263535.90,9014.07,HORIZONTE VIAGENS TURISMO
1076801732,57,262138.61,7876.05,ATLAS IDIOMAS


Lojistas da tabela sem nenhuma linha na Orbe (EC ausente ou lojista só de remessa):


,lojista,ec,obs
2,Horizonte Viagens,NaN,
3,Kiwi Experience,NaN,EC pendente
10,Ponte Aérea Estudantil Ltda,NaN,só remessa
17,Nórdica Educação Internacional Ltda,NaN,só remessa
22,Novo Horizonte Intercambio,NaN,
24,Globo Trotter Viagens Ltda,1157874034,INATIVO desde jul/26


### 3.3 Primeiro cruzamento taxa aplicada × contratada (prévia)
**Atenção:** é uma prévia. Não aplica vigência (a Bridge tem duas linhas) e usa "taxa contratada mais próxima" quando há mais de uma. Serve para ver o tamanho do problema, não para cobrar ninguém.

In [16]:
def pct(x):
    if pd.isna(x):
        return None
    s = str(x)
    v = br_num(s)
    if pd.isna(v):
        return None
    return v if "%" in s else (v * 100 if v < 1 else v)

col_faixa = {"debito": "debito", "credito_a_vista": "cred_vista", "credito_2_6x": "cred_2_6x",
             "credito_7_12x": "cred_7_12x", "pix": "pix", "antecipacao": "antecipacao"}
contratos = {}
for _, r in tx.iterrows():
    if pd.notna(r.ec):
        contratos.setdefault(str(r.ec), []).append(r)

def avalia(r):
    ec = r.cod_estabelecimento
    if ec not in contratos:
        return pd.Series({"status": "EC_sem_cadastro", "taxa_contrato": None, "dif_brl": None})
    cands = [pct(c[col_faixa[r.faixa]]) for c in contratos[ec] if r.faixa in col_faixa]
    cands = [x for x in cands if x is not None]
    if not cands:
        return pd.Series({"status": "contrato_sem_taxa_p/_faixa", "taxa_contrato": None, "dif_brl": None})
    prox = min(cands, key=lambda x: abs(x - r.taxa_aplicada_pct_f))
    ok = abs(prox - r.taxa_aplicada_pct_f) < 0.005
    dif = 0.0 if ok else round(r.valor_bruto_f * (r.taxa_aplicada_pct_f - prox) / 100, 2)
    return pd.Series({"status": "ok" if ok else "DIVERGE", "taxa_contrato": prox, "dif_brl": dif})

av = d.join(d.apply(avalia, axis=1))
print("Status por linha (todas as linhas da Orbe):")
display(av.status.value_counts().to_frame("linhas"))
div = av[av.status == "DIVERGE"]
print("Divergências por EC (soma de dif_brl, + = Orbe cobrou a mais que o contrato):")
nome_tx = tx.drop_duplicates("ec").set_index(tx.drop_duplicates("ec").ec.astype(str)).lojista
resumo_div = (div.groupby("cod_estabelecimento")
              .agg(linhas=("id_transacao", "size"), dif_brl=("dif_brl", "sum"),
                   taxas_aplicadas=("taxa_aplicada_pct_f", lambda s: sorted(s.unique())),
                   taxas_contrato=("taxa_contrato", lambda s: sorted(s.unique())))
              .assign(lojista=lambda x: x.index.map(nome_tx)).round(2))
display(resumo_div)
print("Total preliminar:", round(div.dif_brl.sum(), 2))
div[["id_transacao", "data_movimento", "cod_estabelecimento", "nome_estabelecimento", "tipo_lancamento", "faixa",
     "valor_bruto", "taxa_aplicada_pct", "taxa_contrato", "dif_brl"]].head(40)

Status por linha (todas as linhas da Orbe):


,linhas
status,
ok,1566
EC_sem_cadastro,290
DIVERGE,87


Divergências por EC (soma de dif_brl, + = Orbe cobrou a mais que o contrato):


,linhas,dif_brl,taxas_aplicadas,taxas_contrato,lojista
cod_estabelecimento,,,,,
1074504011,87,1546.28,"[2.99, 3.49, 4.09]","[2.4899999999999998, 2.9899999999999998, 3.5900000000000003]",Rota Global Intercâmbio Ltda


Total preliminar: 1546.28


,id_transacao,data_movimento,cod_estabelecimento,nome_estabelecimento,tipo_lancamento,faixa,valor_bruto,taxa_aplicada_pct,taxa_contrato,dif_brl
725,TX88423890,12/09/2026,1074504011,ROTA GLOBAL INTERCAMBIO,VENDA,credito_7_12x,"6449,10","4,09",3.59,32.25
728,TX88424136,12/09/2026,1074504011,ROTA GLOBAL INTERCAMBIO,VENDA,credito_2_6x,"1882,17","3,49",2.99,9.41
747,TX88423647,12/09/2026,1074504011,ROTA GLOBAL INTERCAMBIO,VENDA,credito_7_12x,"13256,06","4,09",3.59,66.28
751,TX88424022,12/09/2026,1074504011,ROTA GLOBAL INTERCAMBIO,VENDA,credito_a_vista,"1074,84","2,99",2.49,5.37
803,TX88424077,12/09/2026,1074504011,ROTA GLOBAL INTERCAMBIO,VENDA,credito_2_6x,"3568,16","3,49",2.99,17.84
809,TX88425052,13/09/2026,1074504011,ROTA GLOBAL INTERCAMBIO,VENDA,credito_7_12x,"7675,72","4,09",3.59,38.38
820,TX88425173,13/09/2026,1074504011,ROTA GLOBAL INTERCAMBIO,VENDA,credito_2_6x,"2743,40","3,49",2.99,13.72
843,TX88425448,13/09/2026,1074504011,ROTA GLOBAL INTERCAMBIO,VENDA,credito_a_vista,"1998,32","2,99",2.49,9.99
861,TX88427107,14/09/2026,1074504011,ROTA GLOBAL INTERCAMBIO,VENDA,credito_a_vista,"862,46","2,99",2.49,4.31
869,TX88426252,14/09/2026,1074504011,ROTA GLOBAL INTERCAMBIO,VENDA,credito_2_6x,"2552,83","3,49",2.99,12.76


In [17]:
# Rota Global (EC 1074504011): quando a taxa "fora do contrato" começa, e vale para tudo depois disso?
rg = av[(av.cod_estabelecimento == "1074504011") & av.tipo_lancamento.isin(["VENDA", "ESTORNO"])].copy()
rg["dt"] = pd.to_datetime(rg.data_movimento, format="%d/%m/%Y")
primeira = rg.loc[rg.status == "DIVERGE", "dt"].min()
print("Primeira linha divergente:", primeira.date(), "| última:", rg.loc[rg.status == "DIVERGE", "dt"].max().date())
print("Antes dessa data:", rg[rg.dt < primeira].status.value_counts().to_dict())
print("Desde essa data :", rg[rg.dt >= primeira].status.value_counts().to_dict())
print("\nDesde essa data, por faixa × status:")
depois = rg[rg.dt >= primeira]
display(pd.crosstab(depois.faixa, depois.status))
print("\nDivergências por tipo de lançamento (estorno devolve a taxa a mais, por isso o sinal negativo):")
div.groupby("tipo_lancamento").dif_brl.agg(["count", "sum"]).round(2)

Primeira linha divergente: 2026-09-12 | última: 2026-09-30
Antes dessa data: {'ok': 51}
Desde essa data : {'DIVERGE': 87, 'ok': 18}

Desde essa data, por faixa × status:


status,DIVERGE,ok
faixa,,
credito_2_6x,31,0
credito_7_12x,18,0
credito_a_vista,38,1
debito,0,7
pix,0,10



Divergências por tipo de lançamento (estorno devolve a taxa a mais, por isso o sinal negativo):


,count,sum
tipo_lancamento,,
ESTORNO,4,-98.44
VENDA,83,1644.72


In [18]:
# Caso especial: Bridge (EC 1075624862) muda o crédito à vista em 15/09 (2,59% -> 2,19%).
# A prévia acima aceitou qualquer uma das duas taxas, então ela NÃO enxerga erro de vigência. Olhando a data:
b = d[(d.cod_estabelecimento == "1075624862") & (d.faixa == "credito_a_vista")].copy()
b["dt"] = pd.to_datetime(b.data_movimento, format="%d/%m/%Y")
b = b.sort_values("dt")
print("Linhas de crédito à vista da Bridge:", len(b))
print(b.groupby(b.dt >= "2026-09-15").taxa_aplicada_pct_f.value_counts().rename("linhas").to_frame())
fora = b[(b.dt >= "2026-09-15") & (b.taxa_aplicada_pct_f > 2.19)].copy()
fora["dif_preliminar_brl"] = (fora.valor_bruto_f * (fora.taxa_aplicada_pct_f - 2.19) / 100).round(2)
print("\nDepois de 15/09 com 2,59% (deveria ser 2,19%):", len(fora), "linhas | diferença preliminar R$", fora.dif_preliminar_brl.sum().round(2))
print("(vigência pela data_movimento; se a regra for pela data da venda original, os estornos mudam de lado)")
fora[["id_transacao", "data_movimento", "tipo_lancamento", "nome_estabelecimento", "valor_bruto", "taxa_aplicada_pct", "dif_preliminar_brl"]]

Linhas de crédito à vista da Bridge: 22
                           linhas
dt    taxa_aplicada_pct_f        
False 2.59                      8
True  2.19                      9
      2.59                      5

Depois de 15/09 com 2,59% (deveria ser 2,19%): 5 linhas | diferença preliminar R$ 29.95
(vigência pela data_movimento; se a regra for pela data da venda original, os estornos mudam de lado)


,id_transacao,data_movimento,tipo_lancamento,nome_estabelecimento,valor_bruto,taxa_aplicada_pct,dif_preliminar_brl
936,TX88428053,15/09/2026,VENDA,BRIDGE IDIOMAS,"1124,71","2,59",4.50
1044,TX88428930,16/09/2026,VENDA,ESCOLA BRIDGE IDIOMAS,"859,57","2,59",3.44
1004,TX88429199,16/09/2026,VENDA,BRIDGE IDIOMAS,"715,68","2,59",2.86
1087,TX88429931,17/09/2026,VENDA,BRIDGE IDIOMAS,"1987,36","2,59",7.95
1148,TX88431123,18/09/2026,VENDA,ESCOLA BRIDGE IDIOMAS,"2801,20","2,59",11.20


## 4. Plano de contas (ERP Fênix)

In [19]:
plano, enc_plano = read_csv_raw(F["plano"])
print("encoding detectado:", enc_plano, "| contas:", len(plano))
print(plano.TIPO.value_counts().to_dict(), "| aceitam lançamento:", (plano.ACEITA_LANCTO == "S").sum())
for _, r in plano.iterrows():
    marca = "  <- recebe lançamento" if r.ACEITA_LANCTO == "S" else ""
    print("   " * r.COD_CONTA.count(".") + f"{r.COD_CONTA}  {r.DESCRICAO}{marca}")

encoding detectado: cp1252 | contas: 52
{'ANALITICA': 32, 'SINTETICA': 20} | aceitam lançamento: 32
1  ATIVO
   1.1  ATIVO CIRCULANTE
      1.1.01  DISPONIBILIDADES
         1.1.01.001  Banco Aurora C/C 4471-0  <- recebe lançamento
         1.1.01.002  Banco Meridiano C/C 90233-5 (Câmbio)  <- recebe lançamento
      1.1.02  APLICAÇÕES FINANCEIRAS
         1.1.02.001  Aplicação automática Meridiano  <- recebe lançamento
      1.1.03  CRÉDITOS A RECEBER
         1.1.03.001  Recebíveis de adquirente - Orbe  <- recebe lançamento
         1.1.03.002  Comissões de intermediação a receber - câmbio  <- recebe lançamento
         1.1.03.003  Adiantamentos diversos  <- recebe lançamento
      1.1.09  CONTAS TRANSITÓRIAS
         1.1.09.001  Transferências entre contas próprias  <- recebe lançamento
         1.1.09.002  Créditos não identificados  <- recebe lançamento
2  PASSIVO
   2.1  PASSIVO CIRCULANTE
      2.1.01  OBRIGAÇÕES COM LOJISTAS
         2.1.01.001  Repasses a lojistas a pagar  <- r

## 5. Extrato Banco Aurora (conta principal, CSV)
O arquivo tem **cabeçalho de 6 linhas** antes da tabela, valores no formato `1.234,56`, e linhas de saldo no meio e no fim. A coluna `linha` guarda o **número da linha física** (`L9`, `L10`...), que é o `ref_origem` exigido na saída.

In [20]:
txt_au, enc_au = read_text(F["aurora"])
linhas_au = txt_au.splitlines()
hdr_i = next(i for i, l in enumerate(linhas_au) if l.startswith("Data;"))
print("encoding:", enc_au, "| linhas no arquivo:", len(linhas_au), "| cabeçalho da tabela na linha", hdr_i + 1)
print("\nMetadados do topo:")
for l in linhas_au[:hdr_i]:
    print("  ", l)
recs = []
for i in range(hdr_i + 1, len(linhas_au)):
    p = linhas_au[i].split(";")
    p += [""] * (5 - len(p))
    recs.append({"linha": f"L{i + 1}", "data": p[0], "lancamento": p[1], "documento": p[2], "valor": p[3], "saldo": p[4]})
au_all = pd.DataFrame(recs)
print("\nLinhas que não são lançamento:")
display(au_all[~au_all.data.str.match(r"\d{2}/\d{2}/\d{4}") | (au_all.lancamento == "SALDO ANTERIOR")])
au = au_all[au_all.data.str.match(r"\d{2}/\d{2}/\d{4}") & (au_all.lancamento != "SALDO ANTERIOR")].copy()
au["valor_f"] = au.valor.map(br_num)
au["saldo_f"] = au.saldo.map(br_num)
au["tipo"] = au.lancamento.map(prefixo)
print("Lançamentos:", len(au))
au.head(8)

encoding: utf-8 | linhas no arquivo: 95 | cabeçalho da tabela na linha 7

Metadados do topo:
   Banco Aurora S.A.;;;;
   Extrato de Conta Corrente - Pessoa Jurídica;;;;
   Cliente: TARVO PAGAMENTOS LTDA;CNPJ: 84.626.315/0001-39;;;
   Agência: 0318;Conta: 4471-0;;;
   Período: 01/09/2026 a 30/09/2026;;;;
   ;;;;

Linhas que não são lançamento:


,linha,data,lancamento,documento,valor,saldo
0,L8,31/08/2026,SALDO ANTERIOR,,,"612.334,81"
85,L93,,,,,
86,L94,,SALDO FINAL,,,"-1.920.088,60"
87,L95,,Os lançamentos estão sujeitos a confirmação.,,,


Lançamentos: 84


,linha,data,lancamento,documento,valor,saldo,valor_f,saldo_f,tipo
1,L9,01/09/2026,TED RECEBIDA 99.876.669/0001-09 ORBE ADQUIRENCIA SA,988044,"66.026,00","678.360,81",66026.00,678360.81,TED RECEBIDA
2,L10,01/09/2026,TAR PACOTE SERVICOS PJ,,"-389,00","677.971,81",-389.00,677971.81,TAR PACOTE
3,L11,02/09/2026,TED RECEBIDA 99.876.669/0001-09 ORBE ADQUIRENCIA SA,428618,"209.630,03","887.601,84",209630.03,887601.84,TED RECEBIDA
4,L12,02/09/2026,TRANSF ENTRE CONTAS P/ BCO MERIDIANO AG 0091 CC 90233-5,,"-350.000,00","537.601,84",-350000.00,537601.84,TRANSF ENTRE
5,L13,03/09/2026,TED RECEBIDA 99.876.669/0001-09 ORBE ADQUIRENCIA SA,125460,"170.754,70","708.356,54",170754.70,708356.54,TED RECEBIDA
6,L14,04/09/2026,TED RECEBIDA 99.876.669/0001-09 ORBE ADQUIRENCIA SA,828516,"101.224,11","809.580,65",101224.11,809580.65,TED RECEBIDA
7,L15,04/09/2026,PAGTO SALARIOS - FOLHA 08/2026,,"-184.732,18","624.848,47",-184732.18,624848.47,PAGTO SALARIOS
8,L16,05/09/2026,BOLETO CONDOMINIO ED. FARIA LIMA 3311,,"-28.450,00","596.398,47",-28450.00,596398.47,BOLETO CONDOMINIO


In [21]:
saldo_ini = br_num(au_all.loc[au_all.lancamento == "SALDO ANTERIOR", "saldo"].iloc[0])
saldo_fim = br_num(au_all.loc[au_all.lancamento == "SALDO FINAL", "saldo"].iloc[0])
au["saldo_calc"] = (saldo_ini + au.valor_f.cumsum()).round(2)
au["dif_saldo"] = (au.saldo_f - au.saldo_calc).round(2)
print(f"Saldo anterior: {saldo_ini:,.2f} | entradas: {au.valor_f[au.valor_f>0].sum():,.2f} | "
      f"saídas: {au.valor_f[au.valor_f<0].sum():,.2f} | saldo final calculado: {saldo_ini + au.valor_f.sum():,.2f} | "
      f"saldo final informado: {saldo_fim:,.2f}")
print("Linhas onde o saldo informado ≠ saldo recalculado:", int((au.dif_saldo.abs() > 0.01).sum()))
neg = au[au.saldo_f < 0]
print("Primeira linha com saldo negativo:", neg.iloc[0][["linha", "data", "saldo"]].to_dict() if len(neg) else "nenhuma")
print("Menor saldo do mês:", au.saldo_f.min(), "| dias-linha com saldo negativo:", len(neg), "de", len(au))
print("\nLançamentos por tipo (2 primeiras palavras):")
(au.groupby("tipo").agg(linhas=("linha", "size"), total=("valor_f", "sum")).sort_values("total").round(2))

Saldo anterior: 612,334.81 | entradas: 3,576,996.29 | saídas: -6,109,419.70 | saldo final calculado: -1,920,088.60 | saldo final informado: -1,920,088.60
Linhas onde o saldo informado ≠ saldo recalculado: 0
Primeira linha com saldo negativo: {'linha': 'L30', 'data': '09/09/2026', 'saldo': '-100.597,44'}
Menor saldo do mês: -2289532.94 | dias-linha com saldo negativo: 62 de 84

Lançamentos por tipo (2 primeiras palavras):


,linhas,total
tipo,,
PIX ENVIADO,45,-4237067.61
TRANSF ENTRE,4,-1450000.00
PAGTO SALARIOS,1,-184732.18
DEBITO DARF,3,-77198.01
GPS INSS,1,-51320.66
DEB AUT,1,-41208.77
BOLETO CONDOMINIO,1,-28450.00
PAGTO ISS,1,-21877.40
GRF FGTS,1,-14778.57


In [22]:
# Para quem a Aurora fez PIX? (repasses a lojistas?) Heurística só para olhar: 1ª palavra do nome x nomes da tabela.
STOP = {"DE", "DO", "DA", "E", "LTDA", "ESCOLA"}
nomes_tx = {norm(n): n for n in tx.lojista}

def candidatos(nome):
    toks = [t for t in norm(nome).split() if t not in STOP]
    if not toks:
        return []
    return [orig for k, orig in nomes_tx.items() if toks[0] in k.split()]

pix = au[au.lancamento.str.startswith("PIX ENVIADO")].copy()
pix["beneficiario"] = pix.lancamento.str.replace("PIX ENVIADO ", "", regex=False)
ben = pix.groupby("beneficiario").agg(pagamentos=("linha", "size"), total=("valor_f", "sum")).round(2)
ben["candidatos_na_tabela"] = ben.index.map(lambda n: candidatos(n))
ben["n_cand"] = ben.candidatos_na_tabela.map(len)
print("PIX enviados:", len(pix), "| beneficiários distintos:", len(ben), "| total:", round(pix.valor_f.sum(), 2))
print("n_cand = 0 → sem correspondência na tabela; n_cand > 1 → ambíguo")
ben.sort_values(["n_cand", "total"], ascending=[True, True])

PIX enviados: 45 | beneficiários distintos: 23 | total: -4237067.61
n_cand = 0 → sem correspondência na tabela; n_cand > 1 → ambíguo


,pagamentos,total,candidatos_na_tabela,n_cand
beneficiario,,,,
ATLAS IDIOMAS LTDA,2,-249301.77,[],0
EDU ABROAD CONSULTORIA LTD,2,-62037.47,[],0
CONTABILIDADE EXATA LTDA,1,-6800.00,[],0
MAPLE STUDY ASSESSORIA EDU,2,-330030.98,[Maple Study Assessoria Educacional Ltda],1
BEM-VINDO EXCHANGE LTDA,2,-274745.29,[Bem-Vindo Exchange Ltda],1
VELA TURISMO EDUCACIONAL L,2,-265816.58,[Vela],1
SOL NASCENTE VIAGENS LTDA,2,-258461.00,[Sol Nascente Viagens Ltda],1
OXFORD PREP CURSOS LTDA,2,-239547.16,[Oxford Prep Cursos Ltda],1
CANGURU INTERCAMBIOS LTDA,2,-234977.66,[Canguru],1


In [23]:
print("Lançamentos de interesse (não são PIX de repasse nem TED da Orbe):")
outros = au[~au.lancamento.str.startswith(("PIX ENVIADO", "TED RECEBIDA 99.876.669"))]
outros[["linha", "data", "lancamento", "documento", "valor", "saldo"]]

Lançamentos de interesse (não são PIX de repasse nem TED da Orbe):


,linha,data,lancamento,documento,valor,saldo
2,L10,01/09/2026,TAR PACOTE SERVICOS PJ,,"-389,00","677.971,81"
4,L12,02/09/2026,TRANSF ENTRE CONTAS P/ BCO MERIDIANO AG 0091 CC 90233-5,,"-350.000,00","537.601,84"
7,L15,04/09/2026,PAGTO SALARIOS - FOLHA 08/2026,,"-184.732,18","624.848,47"
8,L16,05/09/2026,BOLETO CONDOMINIO ED. FARIA LIMA 3311,,"-28.450,00","596.398,47"
34,L42,10/09/2026,PAGTO ISS SAO PAULO DAMSP 08/2026,,"-21.877,40","-1.041.759,94"
36,L44,11/09/2026,TRANSF ENTRE CONTAS P/ BCO MERIDIANO AG 0091 CC 90233-5,,"-420.000,00","-1.276.574,84"
39,L47,15/09/2026,DEB AUT AMAZON WEB SERVICES BR,,"-41.208,77","-1.027.062,67"
40,L48,15/09/2026,CARTAO PJ - GOOGLE WORKSPACE,,"-2.210,00","-1.029.272,67"
42,L50,16/09/2026,PIX RECEBIDO 38686002000180 SEM IDENTIFICACAO,,"7.350,00","-752.669,98"
46,L54,18/09/2026,GPS INSS 08/2026,,"-51.320,66","-455.932,33"


## 6. Extrato Banco Meridiano (conta de câmbio, OFX)
OFX em SGML (tags sem fechamento), declara `ENCODING:USASCII` mas `CHARSET:1252`. O identificador de linha é o `FITID`.

In [24]:
txt_me, enc_me = read_text(F["meridiano"])
print("encoding efetivo:", enc_me)
print("Cabeçalho OFX:", re.findall(r"^(\w+):(\S+)", txt_me[:200], flags=re.M))
conta = {k: re.search(rf"<{k}>([^\r\n<]*)", txt_me).group(1) for k in ["BANKID", "BRANCHID", "ACCTID", "ACCTTYPE", "DTSTART", "DTEND"]}
print("Conta:", conta)
blocos = re.findall(r"<STMTTRN>(.*?)</STMTTRN>", txt_me, flags=re.S)
mer = pd.DataFrame([dict(re.findall(r"<(\w+)>([^\r\n<]*)", b)) for b in blocos])
mer["data"] = mer.DTPOSTED.str[:8]
mer["valor"] = mer.TRNAMT.astype(float)
mer["tipo"] = mer.MEMO.map(prefixo)
bal = re.search(r"<LEDGERBAL>\s*<BALAMT>([-\d.]+)", txt_me)
print("Lançamentos:", len(mer), "| FITID duplicado:", mer.FITID.duplicated().sum(),
      "| saldo final (LEDGERBAL):", bal.group(1) if bal else None)
print("CREDIT com valor ≤ 0:", ((mer.TRNTYPE == "CREDIT") & (mer.valor <= 0)).sum(),
      "| DEBIT com valor ≥ 0:", ((mer.TRNTYPE == "DEBIT") & (mer.valor >= 0)).sum())
print("Entradas:", round(mer.valor[mer.valor > 0].sum(), 2), "| Saídas:", round(mer.valor[mer.valor < 0].sum(), 2),
      "| movimento líquido:", round(mer.valor.sum(), 2))
print("(o OFX não traz saldo inicial, então o saldo final não pode ser recalculado só com este arquivo)")
mer[["FITID", "data", "TRNTYPE", "valor", "MEMO", "CHECKNUM"]].head(8)

encoding efetivo: cp1252
Cabeçalho OFX: [('OFXHEADER', '100'), ('DATA', 'OFXSGML'), ('VERSION', '102'), ('SECURITY', 'NONE'), ('ENCODING', 'USASCII'), ('CHARSET', '1252'), ('COMPRESSION', 'NONE'), ('OLDFILEUID', 'NONE'), ('NEWFILEUID', 'NONE')]
Conta: {'BANKID': '0477', 'BRANCHID': '0091', 'ACCTID': '90233-5', 'ACCTTYPE': 'CHECKING', 'DTSTART': '20260901000000[-3:BRT]', 'DTEND': '20260930235959[-3:BRT]'}
Lançamentos: 100 | FITID duplicado: 0 | saldo final (LEDGERBAL): 1320453.26
CREDIT com valor ≤ 0: 0 | DEBIT com valor ≥ 0: 0
Entradas: 7318364.9 | Saídas: -6202029.67 | movimento líquido: 1116335.23
(o OFX não traz saldo inicial, então o saldo final não pode ser recalculado só com este arquivo)


,FITID,data,TRNTYPE,valor,MEMO,CHECKNUM
0,MRD2026090003,20260901,CREDIT,207215.87,PIX RECEB LISBOA CURSOS E VIAGENS,288643
1,MRD2026090008,20260901,CREDIT,115004.93,PIX RECEB MUNDO ABERTO INTERCÂMBIO,121551
2,MRD2026090016,20260901,CREDIT,72488.87,PIX RECEB NORDICA EDUCACAO INTERNA,124148
3,MRD2026090022,20260901,CREDIT,40011.58,PIX RECEB DUBLIN HUB INTERCÂMBIO L,959584
4,MRD2026090034,20260901,CREDIT,112819.08,PIX RECEB CASA DO ESTUDANTE VIAGEN,987956
5,MRD2026090036,20260901,CREDIT,170670.73,PIX RECEB EDU ABROAD CONSULTORIA L,918960
6,MRD2026090047,20260901,CREDIT,216777.54,PIX RECEB ROTA GLOBAL INTERCÂMBIO,565710
7,MRD2026090055,20260901,CREDIT,16409.36,PIX RECEB KIWI EXPERIENCE TURISMO,417016


In [25]:
print("Lançamentos por tipo (2 primeiras palavras do MEMO):")
(mer.groupby("tipo").agg(linhas=("FITID", "size"), total=("valor", "sum")).sort_values("total").round(2))

Lançamentos por tipo (2 primeiras palavras do MEMO):


,linhas,total
tipo,,
TED ENVIADA,21,-5611447.12
APLICACAO AUTOMATICA,3,-588124.00
SEGURO EMPRESARIAL,1,-1830.44
TARIFA PACOTE,1,-450.00
TARIFA MANUTENCAO,1,-129.90
IOF S,1,-48.21
RENDIMENTO APLIC,1,3912.66
TED RECEB,1,61884.12
RESGATE AUTOMATICO,3,472881.00


In [26]:
rec = mer[mer.MEMO.str.startswith("PIX RECEB")].copy()
rec["pagador"] = rec.MEMO.str.replace("PIX RECEB ", "", regex=False).str.strip()
print("Nome do pagador no MEMO tem no máximo", rec.pagador.str.len().max(), "caracteres (vem cortado)")
print("Quem manda PIX para o Meridiano (recursos de remessa?):")
rec.groupby("pagador").agg(pix=("FITID", "size"), total=("valor", "sum")).round(2).sort_values("total", ascending=False)

Nome do pagador no MEMO tem no máximo 24 caracteres (vem cortado)
Quem manda PIX para o Meridiano (recursos de remessa?):


,pix,total
pagador,,
ROTA GLOBAL INTERCÂMBIO,5,668059.95
MAPLE STUDY ASSESSORIA E,5,583530.13
CANGURU INTERCÂMBIOS LTD,5,543693.33
MUNDO ABERTO INTERCÂMBIO,5,481950.02
NORDICA EDUCACAO INTERNA,5,473909.27
PONTE AÉREA ESTUDANTIL L,5,452249.03
CASA DO ESTUDANTE VIAGEN,4,417843.39
LISBOA CURSOS E VIAGENS,5,407463.55
EDU ABROAD CONSULTORIA L,5,394222.69


## 7. Cruzamentos entre bancos e com a Orbe
### 7.1 Transferências entre contas próprias: Aurora → Meridiano

In [27]:
a_tr = au[au.lancamento.str.startswith("TRANSF ENTRE CONTAS")][["linha", "data", "valor_f"]].copy()
a_tr["dt"] = pd.to_datetime(a_tr.data, format="%d/%m/%Y").dt.strftime("%Y%m%d")
a_tr["chave_valor"] = a_tr.valor_f.abs().round(2)
m_tr = mer[mer.MEMO.str.contains("MESMA TITULARIDADE")][["FITID", "data", "valor"]].copy()
m_tr["chave_valor"] = m_tr.valor.abs().round(2)
cruz = a_tr.merge(m_tr, left_on=["dt", "chave_valor"], right_on=["data", "chave_valor"], how="outer",
                  suffixes=("_aurora", "_meridiano"), indicator=True)
cruz[["linha", "data_aurora", "valor_f", "FITID", "data_meridiano", "valor", "_merge"]]

,linha,data_aurora,valor_f,FITID,data_meridiano,valor,_merge
0,L12,02/09/2026,-350000.0,MRD2026090107,20260902,350000.0,both
1,L44,11/09/2026,-420000.0,MRD2026090274,20260911,420000.0,both
2,L58,21/09/2026,-380000.0,MRD2026090454,20260921,380000.0,both
3,L92,30/09/2026,-300000.0,NaN,NaN,NaN,left_only


### 7.2 TEDs que a Orbe pagou na Aurora × o que a Orbe reporta por dia
Não sei a regra de liquidação (D+0, D+30?). Mostro as duas séries lado a lado, **sem concluir nada**: é pergunta ao financeiro.

In [28]:
ted = au[au.lancamento.str.contains("ORBE")][["linha", "data", "valor_f"]].rename(columns={"valor_f": "ted_aurora"})
dia = (d.assign(grupo=d.tipo_lancamento.map(lambda t: "vendas_estornos" if t in ("VENDA", "ESTORNO") else "antecipacoes"))
         .pivot_table(index="data_movimento", columns="grupo", values="valor_liquido_f", aggfunc="sum").round(2))
lado = dia.reset_index().merge(ted, left_on="data_movimento", right_on="data", how="outer")
lado["dt"] = pd.to_datetime(lado.data_movimento.fillna(lado.data), format="%d/%m/%Y")
lado = lado.sort_values("dt")[["dt", "vendas_estornos", "antecipacoes", "linha", "ted_aurora"]]
print("TEDs da Orbe na Aurora:", len(ted), "| soma:", round(ted.ted_aurora.sum(), 2),
      "| líquido total reportado pela Orbe (vendas+estornos):", round(dia.vendas_estornos.sum(), 2))
lado

TEDs da Orbe na Aurora: 21 | soma: 3557246.29 | líquido total reportado pela Orbe (vendas+estornos): 6125403.55


,dt,vendas_estornos,antecipacoes,linha,ted_aurora
0,2026-09-01,229928.86,31098.06,L9,66026.00
1,2026-09-02,222432.30,47033.50,L11,209630.03
2,2026-09-03,173159.76,27477.23,L13,170754.70
3,2026-09-04,153813.55,97398.27,L14,101224.11
4,2026-09-05,196963.31,-98249.12,NaN,NaN
5,2026-09-06,274708.12,NaN,NaN,NaN
6,2026-09-07,159315.04,NaN,NaN,NaN
7,2026-09-08,253780.61,39181.13,L17,119663.57
8,2026-09-09,246771.97,16468.75,L25,206834.86
9,2026-09-10,193451.57,-9724.52,L31,165039.27


## 8. Ponte Câmbio — e-mail e relatório de remessas

In [29]:
msg = email.message_from_bytes(F["ponte"].read_bytes(), policy=policy.default)
for h in ["From", "To", "Cc", "Date", "Subject"]:
    print(f"{h}: {msg[h]}")
print("\n--- corpo ---")
print(msg.get_body(preferencelist=("plain",)).get_content())
anexos = [(p.get_filename(), p.get_content()) for p in msg.iter_attachments()]
print("Anexos:", [(n, len(b)) for n, b in anexos])

From: Relatórios Ponte Câmbio <relatorios@pontecambio.com.br>
To: financeiro@tarvo.com.br
Cc: comercial.parceiros@pontecambio.com.br
Date: Fri, 02 Oct 2026 07:15:03 -0300
Subject: Relatório mensal de operações - TARVO - 09/2026

--- corpo ---
Prezados,

Segue em anexo o relatório mensal de operações de câmbio do parceiro TARVO PAGAMENTOS referente à competência 09/2026.

Conforme contrato de parceria, o repasse da participação de 50% (cinquenta por cento) do spread realizado será creditado em 06/10/2026, mediante recebimento da nota fiscal de intermediação.

Lembramos que operações com status "APROVADO C/ RESSALVA" seguem normalmente para liquidação.

Em caso de divergência, favor responder este e-mail em até 5 dias úteis.

Atenciosamente,
Backoffice Parceiros - Ponte Câmbio S.A. - CNPJ 75.374.179/0001-46
Este é um e-mail automático.

Anexos: [('PonteCambio_Relatorio_TARVO_092026.xlsx', 30021)]


In [30]:
nome_anexo, bytes_anexo = anexos[0]
grids_pt, coment_pt = read_xlsx_raw(bytes_anexo)
print("Abas:", list(grids_pt), "| comentários:", coment_pt)
gp = grids_pt["Operações"]
print("\nTopo da aba Operações:")
display(gp.loc[1:4].dropna(how="all", axis=1))
cols_op = ["num_op", "dt_contratacao", "dt_liquidacao", "cnpj_cliente", "cliente", "moeda", "valor_me", "taxa_cliente",
           "ptax", "valor_brl", "spread_brl", "spread_pct", "spread_estimado_brl", "iof_brl", "status_compliance", "observacao"]
print("Cabeçalho original (linha 5):", list(gp.loc[5]))
bruto_op = gp.loc[6:].reindex(columns=list(gp.columns))
if bruto_op.shape[1] == len(cols_op):
    bruto_op.columns = cols_op
eh_op = bruto_op.num_op.astype(str).str.match(r"PC-\d{4}-\d+")
print("\nLinhas que NÃO são operação (rodapé, totais, vazias):")
display(bruto_op[~eh_op].dropna(how="all"))
op = bruto_op[eh_op].reset_index(drop=True)
for c in ["valor_me", "taxa_cliente", "ptax", "valor_brl", "spread_brl", "spread_pct", "spread_estimado_brl", "iof_brl"]:
    op[c + "_f"] = pd.to_numeric(op[c], errors="coerce")
op["cnpj14"] = op.cnpj_cliente.map(cnpj14)
print("Operações:", len(op))
op.head(6)

Abas: ['Operações', 'Resumo'] | comentários: []

Topo da aba Operações:


,A
1,PONTE CÂMBIO S.A. - Relatório Mensal de Operações por Parceiro
2,Parceiro: TARVO PAGAMENTOS LTDA | CNPJ 84.626.315/0001-39
3,Competência: 09/2026 | Gerado em 02/10/2026 07:12


Cabeçalho original (linha 5): ['Nº Operação', 'Data Contratação', 'Data Liquidação', 'CNPJ Cliente', 'Cliente', 'Moeda', 'Valor ME', 'Taxa Cliente', 'PTAX Ref.', 'Valor R$', 'Spread R$', 'Spread %', 'Spread Estimado R$ (D-1)', 'IOF R$', 'Status Compliance', 'Observação']

Linhas que NÃO são operação (rodapé, totais, vazias):


,num_op,dt_contratacao,dt_liquidacao,cnpj_cliente,cliente,moeda,valor_me,taxa_cliente,ptax,valor_brl,spread_brl,spread_pct,spread_estimado_brl,iof_brl,status_compliance,observacao
240,Total de operações:,233,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
241,Spread total R$:,78073.59,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Operações: 233


,num_op,dt_contratacao,dt_liquidacao,cnpj_cliente,cliente,moeda,valor_me,taxa_cliente,ptax,valor_brl,spread_brl,spread_pct,spread_estimado_brl,iof_brl,status_compliance,observacao,valor_me_f,taxa_cliente_f,ptax_f,valor_brl_f,spread_brl_f,spread_pct_f,spread_estimado_brl_f,iof_brl_f,cnpj14
0,PC-0926-0002,46295,46296,34671260000100,MAPLE STUDY ASSESSOR,NZD,5402.22,3.2796,3.2311,17717.12,262.01,0.01501,279.09,67.33,APROVADO,,5402.22,3.2796,3.2311,17717.12,262.01,0.015010,279.09,67.33,34671260000100
1,PC-0926-0003,46275,46276,14.760.206/0001-24,NÓRDICA EDUCAÇÃO INT,USD,7929.38,5.5281,5.4599,43834.41,540.78,0.012491,455.08,166.57,APROVADO,,7929.38,5.5281,5.4599,43834.41,540.78,0.012491,455.08,166.57,14760206000124
2,PC-0926-0004,46267,46268,61.720.397/0001-80,LISBOA CURSOS E VIAG,USD,5733.03,5.4114,5.3262,31023.72,488.45,0.015996,480.85,117.89,APROVADO,,5733.03,5.4114,5.3262,31023.72,488.45,0.015996,480.85,117.89,61720397000180
3,PC-0926-0007,46294,46295,14.760.206/0001-24,NÓRDICA EDUCAÇÃO INT,CAD,5289.9,4.0096,3.9601,21210.38,261.85,0.0125,346.59,80.59999999999999,APROVADO,,5289.90,4.0096,3.9601,21210.38,261.85,0.012500,346.59,80.60,14760206000124
4,PC-0926-0011,46269,46272,17217888000111,MUNDO ABERTO INTERCÂ,CAD,2438.46,4.092,4.0315,9978.18,147.53,0.015007,99.27,37.92,APROVADO,,2438.46,4.0920,4.0315,9978.18,147.53,0.015007,99.27,37.92,17217888000111
5,PC-0926-0012,46293,46294,34671260000100,MAPLE STUDY ASSESSOR,GBP,3482.69,7.3607,7.2519,25635.04,378.92,0.015003,309.44,97.41,APROVADO,,3482.69,7.3607,7.2519,25635.04,378.92,0.015003,309.44,97.41,34671260000100


In [31]:
print("Status de compliance:")
display(op.status_compliance.value_counts().to_frame("operações"))
com_obs = op[op.observacao.fillna("").str.strip() != ""]
print("Operações com texto na coluna Observação:", len(com_obs))
display(com_obs[["num_op", "cliente", "moeda", "valor_me", "spread_brl", "status_compliance", "observacao"]])
ress = op[op.status_compliance != "APROVADO"]
print(f"Operações 'com ressalva': {len(ress)} | spread somado: R$ {ress.spread_brl_f.sum():,.2f} (sem observação explicando o motivo)")
display(ress[["num_op", "cliente", "moeda", "valor_me", "spread_brl", "status_compliance", "observacao"]])

dup_num = op[op.num_op.duplicated(keep=False)]
identicas = dup_num[cols_op].drop_duplicates().shape[0] == dup_num.num_op.nunique()
print("\nLinhas com Nº de operação repetido:", len(dup_num), "| as cópias são idênticas?", identicas)
print("Spread contado em dobro (R$):", op[op.num_op.duplicated(keep="first")].spread_brl_f.sum())
display(dup_num[cols_op].T)   # transposto: mostra campo a campo o que difere entre as duas cópias
chave = ["cnpj14", "moeda", "valor_me", "dt_contratacao"]
mesma_op = op[op.duplicated(chave, keep=False) & ~op.num_op.duplicated(keep=False)]
print("Mesma operação (cliente, moeda, valor, data) com Nº DIFERENTE:", len(mesma_op))
print("\nSpread ≤ 0:", int((op.spread_brl_f <= 0).sum()),
      "(taxa cliente abaixo da PTAX; o padrão das demais é +1,25% a +1,6%)")
display(op[op.spread_brl_f <= 0][["num_op", "cliente", "moeda", "valor_me", "taxa_cliente", "ptax", "valor_brl", "spread_brl", "spread_pct"]])
print("Datas que não são número serial do Excel:", int(pd.to_numeric(op.dt_contratacao, errors="coerce").isna().sum()))
op["dt_contr"] = pd.to_datetime(pd.to_numeric(op.dt_contratacao, errors="coerce"), unit="D", origin="1899-12-30")
print("Período de contratação:", op.dt_contr.min().date(), "→", op.dt_contr.max().date())

Status de compliance:


,operações
status_compliance,
APROVADO,217
APROVADO C/ RESSALVA,16


Operações com texto na coluna Observação: 5


,num_op,cliente,moeda,valor_me,spread_brl,status_compliance,observacao
71,PC-0926-0162,BAGAGEM LIVRE TURISM,USD,10429.99,821.88,APROVADO,reprocessada
90,PC-0926-0210,PONTE AÉREA ESTUDANT,EUR,7087.81,-687.52,APROVADO,Taxa travada no pedido (D-1)
135,PC-0926-0326,CANGURU INTERCÂMBIOS,CAD,8213.299999999999,-509.22,APROVADO,Taxa travada no pedido (D-1)
199,PC-0926-0491,DUBLIN HUB INTERCÂMB,NZD,5904.65,-292.28,APROVADO,Taxa travada no pedido (D-1)
222,PC-0926-0545,NOVO HORIZONTE INTER,AUD,5918.24,-345.03,APROVADO,Taxa travada no pedido (D-1)


Operações 'com ressalva': 16 | spread somado: R$ 4,829.37 (sem observação explicando o motivo)


...


Linhas com Nº de operação repetido: 2 | as cópias são idênticas? False
Spread contado em dobro (R$): 821.88


,70,71
num_op,PC-0926-0162,PC-0926-0162
dt_contratacao,46287,46287
dt_liquidacao,46288,46288
cnpj_cliente,50900987000179,50900987000179
cliente,BAGAGEM LIVRE TURISM,BAGAGEM LIVRE TURISM
moeda,USD,USD
valor_me,10429.99,10429.99
taxa_cliente,5.3308,5.3308
ptax,5.252,5.252
valor_brl,55600.19,55600.19


Mesma operação (cliente, moeda, valor, data) com Nº DIFERENTE: 0

Spread ≤ 0: 4 (taxa cliente abaixo da PTAX; o padrão das demais é +1,25% a +1,6%)


,num_op,cliente,moeda,valor_me,taxa_cliente,ptax,valor_brl,spread_brl,spread_pct
90,PC-0926-0210,PONTE AÉREA ESTUDANT,EUR,7087.81,6.1518,6.2488,43602.79,-687.52,-0.015523
135,PC-0926-0326,CANGURU INTERCÂMBIOS,CAD,8213.299999999999,3.9321,3.9941,32295.52,-509.22,-0.015523
199,PC-0926-0491,DUBLIN HUB INTERCÂMB,NZD,5904.65,3.1415,3.191,18549.46,-292.28,-0.015512
222,PC-0926-0545,NOVO HORIZONTE INTER,AUD,5918.24,3.4765,3.5348,20574.76,-345.03,-0.016493


Datas que não são número serial do Excel: 0
Período de contratação: 2026-09-01 → 2026-09-30


### 8.1 A conta da Ponte fecha?
Hipótese de fórmula (a confirmar): `valor R$ = valor ME × taxa cliente`, `spread R$ = valor ME × (taxa cliente − PTAX)`, `spread % = (taxa cliente − PTAX) / PTAX`.

In [32]:
op["valor_brl_calc"] = (op.valor_me_f * op.taxa_cliente_f).round(2)
op["spread_calc"] = (op.valor_me_f * (op.taxa_cliente_f - op.ptax_f)).round(2)
op["spread_pct_calc"] = ((op.taxa_cliente_f - op.ptax_f) / op.ptax_f).round(5)
op["dif_valor"] = (op.valor_brl_f - op.valor_brl_calc).round(2)
op["dif_spread"] = (op.spread_brl_f - op.spread_calc).round(2)
op["dif_spread_pct"] = (op.spread_pct_f - op.spread_pct_calc).round(5)
print("Operações com |dif valor R$| > 0,05:", int((op.dif_valor.abs() > 0.05).sum()),
      "| |dif spread R$| > 0,05:", int((op.dif_spread.abs() > 0.05).sum()),
      "| |dif spread %| > 0,0001:", int((op.dif_spread_pct.abs() > 0.0001).sum()))
display(op[(op.dif_valor.abs() > 0.05) | (op.dif_spread.abs() > 0.05) | (op.dif_spread_pct.abs() > 0.0001)]
        [["num_op", "cliente", "moeda", "valor_me", "taxa_cliente", "ptax", "valor_brl", "valor_brl_calc",
          "spread_brl", "spread_calc", "spread_pct", "spread_pct_calc"]])
print("Spread realizado × estimado D-1 (diferença em R$):")
op["dif_real_estimado"] = op.spread_brl_f - op.spread_estimado_brl_f
display(op.dif_real_estimado.describe().round(2).to_frame("R$"))

Operações com |dif valor R$| > 0,05: 0 | |dif spread R$| > 0,05: 0 | |dif spread %| > 0,0001: 0


,num_op,cliente,moeda,valor_me,taxa_cliente,ptax,valor_brl,valor_brl_calc,spread_brl,spread_calc,spread_pct,spread_pct_calc


Spread realizado × estimado D-1 (diferença em R$):


,R$
count,233.00
mean,-12.10
std,144.43
min,-1275.49
25%,-46.44
50%,2.18
75%,45.13
max,320.02


### 8.2 Spread da Ponte × spread contratado na tabela

In [33]:
tx["cnpj14"] = tx.cnpj.map(cnpj14)
tx["spread_f"] = tx.spread_remessa.map(pct)
tab_c = tx.drop_duplicates("cnpj14").set_index("cnpj14")[["lojista", "spread_f"]]
op_u = op.drop_duplicates("num_op")   # sem a cópia idêntica da operação repetida
pc = (op_u.groupby("cnpj14")
        .agg(cliente_ponte=("cliente", "first"), n_ops=("num_op", "size"),
             pct_mediana=("spread_pct_f", "median"), pct_min=("spread_pct_f", "min"), pct_max=("spread_pct_f", "max"),
             spread_brl=("spread_brl_f", "sum")))
for c in ["pct_mediana", "pct_min", "pct_max"]:
    pc[c] = (pc[c] * 100).round(3)
pc = pc.join(tab_c, how="left").rename(columns={"spread_f": "spread_contratado_pct", "lojista": "lojista_na_tabela"})
pc["dif_pp"] = (pc.pct_mediana - pc.spread_contratado_pct).round(3)
pc["situacao"] = pc.apply(lambda r: "CNPJ fora da tabela" if pd.isna(r.lojista_na_tabela)
                          else ("spread em branco na tabela" if pd.isna(r.spread_contratado_pct)
                                else ("DIFERENTE" if abs(r.dif_pp) > 0.05 else "ok")), axis=1)
print("Mediana (e não média) porque os spreads negativos puxam a média para baixo.")
print(pc.situacao.value_counts().to_dict())
pc.sort_values(["situacao", "cliente_ponte"])

Mediana (e não média) porque os spreads negativos puxam a média para baixo.
{'ok': 10, 'spread em branco na tabela': 3, 'DIFERENTE': 1}


,cliente_ponte,n_ops,pct_mediana,pct_min,pct_max,spread_brl,lojista_na_tabela,spread_contratado_pct,dif_pp,situacao
cnpj14,,,,,,,,,,
17217888000111,MUNDO ABERTO INTERCÂ,23,1.50,1.499,1.501,7095.25,Mundo Aberto,1.20,0.3,DIFERENTE
50900987000179,BAGAGEM LIVRE TURISM,8,1.50,1.499,1.501,2585.33,Bagagem Livre Turismo Ltda,1.50,0.0,ok
67869675000142,BEM-VINDO EXCHANGE L,2,1.50,1.500,1.500,726.44,Bem-Vindo Exchange Ltda,1.50,0.0,ok
00520815000100,CASA DO ESTUDANTE VI,18,1.50,1.499,1.501,6151.74,Casa do Estudante Viagens Ltda,1.50,0.0,ok
68567965000102,DUBLIN HUB INTERCÂMB,14,1.50,-1.551,1.501,3252.87,Dublin Hub Intercâmbio Ltda,1.50,0.0,ok
16181276000153,EDU ABROAD CONSULTOR,16,1.30,1.299,1.301,5040.46,EduAbroad,1.30,0.0,ok
61720397000180,LISBOA CURSOS E VIAG,16,1.60,1.599,1.602,6392.62,Lisboa Cursos e Viagens Ltda,1.60,0.0,ok
34671260000100,MAPLE STUDY ASSESSOR,25,1.50,1.498,1.501,8589.91,Maple Study Assessoria Educacional Ltda,1.50,0.0,ok
66564512000199,NOVO HORIZONTE INTER,11,1.40,-1.649,1.401,4069.86,Novo Horizonte Intercambio,1.40,-0.0,ok


### 8.3 A aba Resumo bate com as operações?

In [34]:
rs = grids_pt["Resumo"].copy()
rs.columns = rs.loc[1].tolist()
rs = rs.loc[2:].rename(columns={"Moeda": "moeda"})
for c in ["Qtd", "Volume ME", "Spread R$"]:
    rs[c] = pd.to_numeric(rs[c], errors="coerce")
calc = op.groupby("moeda").agg(qtd_lidas=("num_op", "size"), spread_lidas=("spread_brl_f", "sum")).round(2)
calc_u = op_u.groupby("moeda").agg(qtd_sem_dup=("num_op", "size"), spread_sem_dup=("spread_brl_f", "sum")).round(2)
cmp_ = rs.set_index("moeda")[["Qtd", "Spread R$"]].join(calc).join(calc_u)
cmp_["dif_spread_vs_lidas"] = (cmp_["Spread R$"] - cmp_.spread_lidas).round(2)
cmp_["dif_spread_vs_sem_dup"] = (cmp_["Spread R$"] - cmp_.spread_sem_dup).round(2)
print("Spread total no Resumo:", round(rs["Spread R$"].sum(), 2),
      "| soma das linhas lidas:", round(op.spread_brl_f.sum(), 2),
      "| soma sem a cópia duplicada:", round(op_u.spread_brl_f.sum(), 2))
print("Operações no Resumo:", int(rs["Qtd"].sum()), "| linhas lidas:", len(op), "| únicas:", len(op_u))
if len(op) != len(op_u) and round(rs["Spread R$"].sum(), 2) == round(op.spread_brl_f.sum(), 2):
    print("→ o Resumo soma a operação repetida duas vezes (tem o spread da cópia duplicada).")
cmp_

Spread total no Resumo: 78073.59 | soma das linhas lidas: 78073.59 | soma sem a cópia duplicada: 77251.71
Operações no Resumo: 233 | linhas lidas: 233 | únicas: 232


→ o Resumo soma a operação repetida duas vezes (tem o spread da cópia duplicada).

,Qtd,Spread R$,qtd_lidas,spread_lidas,qtd_sem_dup,spread_sem_dup,dif_spread_vs_lidas,dif_spread_vs_sem_dup
moeda,,,,,,,,
USD,37,14007.38,37,14007.38,36,13185.50,0.0,821.88
EUR,33,16074.48,33,16074.48,33,16074.48,0.0,0.00
GBP,36,17739.02,36,17739.02,36,17739.02,0.0,0.00
CAD,42,10705.21,42,10705.21,42,10705.21,0.0,0.00
AUD,48,11329.45,48,11329.45,48,11329.45,0.0,0.00
NZD,37,8218.05,37,8218.05,37,8218.05,0.0,0.00


## 9. O que a exploração levanta
Resumo do que as células acima mostraram com o data pack de setembro/2026. Cada item vira **premissa** em `HIPOTESES.md` ou **pergunta ao financeiro** (quando muda número). Os valores em R$ de divergência são **prévias**: não aplicam todas as regras de vigência e não passaram por validação do financeiro.

### Orbe (checkout)
- **1.943 linhas** (1.840 vendas, 59 estornos, 38 antecipações, 6 reversões), 22 ECs. 8 ECs aparecem com mais de um nome, e nenhum nome aparece em dois ECs: **o EC é chave segura, o nome não**.
- **A conta fecha em todas as linhas**, mas a Orbe **trunca** o `valor_taxa` em centavos (reproduz 1.943 de 1.943 linhas; arredondar acerta só 942). Para recalcular diferença em R$, usar truncamento, senão aparecem ±R$ 0,01 falsos.
- **4 ECs sem contrato verificável (290 linhas)**: Atlas Idiomas (57 linhas, não existe na tabela), Kiwi (85, "EC pendente"), Horizonte Viagens (75) e Horizonte Intercâmbio (73). Os três últimos têm lojista parecido na tabela, mas com EC em branco; ligá-los por nome é palpite.
- **59 estornos**: 50 têm a venda original no arquivo; **9 referenciam vendas de agosto** (sem origem no mês).
- **Antecipações**: 44 linhas, todas a 2,10%. As 6 reversões têm a antecipação original no arquivo.
- **Divergência de taxa (prévia): Rota Global.** A partir de 12/09, todas as linhas de crédito (87) vêm com +0,50 p.p. sobre o contrato (2,99/3,49/4,09 em vez de 2,49/2,99/3,59); débito e Pix seguem corretos. Prévia: R$ 1.546,28 (vendas +R$ 1.644,72; estornos −R$ 98,44).
- **Vigência da Bridge**: depois de 15/09 o crédito à vista deveria ser 2,19%, mas 5 linhas ainda vêm a 2,59% (prévia R$ 29,95). A comparação por "taxa mais próxima" não enxerga isso; **vigência precisa ser aplicada por data**.

### Tabela de taxas
- Comentário de célula na linha da Casa do Estudante: *"conversar com Diego (comercial) sobre condição especial"*.
- Percentuais em dois formatos (decimal e texto `1,49%`), CNPJs com e sem pontuação, **2 CNPJs com 13 dígitos** (Fluente e Campus Global, zero à esquerda perdido), Canguru com spread escrito "padrão", 13 spreads em branco, CNPJ da Bridge repetido em duas linhas (vigências).

### Bancos
- **Aurora**: os saldos informados batem com o recálculo (0 diferenças), mas a conta fica **negativa desde 09/09** (L30), chega a **−R$ 2.289.532,94** e fecha em −R$ 1.920.088,60. Saíram R$ 4,24 mi em PIX para 23 beneficiários contra R$ 3,56 mi em TEDs da Orbe.
- **Transferências para o Meridiano**: 4 saídas (R$ 1,45 mi). 3 aparecem no OFX; **a de 30/09 (R$ 300.000, L92) não tem entrada correspondente**.
- **TEDs da Orbe × relatório**: o líquido de vendas e estornos reportado é R$ 6,13 mi contra R$ 3,56 mi de TEDs, e não há correspondência dia a dia. A regra de liquidação não está em nenhum arquivo.
- **Nome não serve de chave**: "EDU ABROAD CONSULTORIA" × "EduAbroad", "HORIZONTE VIAGENS E TURISM" casa com dois lojistas, "ATLAS IDIOMAS" não casa com ninguém.
- **Meridiano (OFX)**: 100 lançamentos, nome do pagador cortado em 24 caracteres, saldo final não recalculável (o arquivo não traz saldo inicial). Plano de contas em `cp1252` (32 contas analíticas).

### Ponte Câmbio (remessas)
- **233 linhas, 232 únicas**: `PC-0926-0162` aparece duas vezes e a segunda cópia tem a observação *"reprocessada"*. O Resumo e o rodapé do relatório (233 operações, R$ 78.073,59) **contam as duas**; sem a cópia, R$ 77.251,71 (R$ 821,88 em dobro).
- **4 spreads negativos**, todos com a observação *"Taxa travada no pedido (D-1)"* (Ponte Aérea, Canguru, Dublin Hub, Novo Horizonte). Há explicação, mas não está claro quem absorve o prejuízo.
- **16 operações "APROVADO C/ RESSALVA"** (spread somado R$ 4.829,37), sem texto de motivo.
- **As contas da Ponte fecham** (`valor R$ = ME × taxa`, `spread R$ = ME × (taxa − PTAX)`, `spread % = (taxa − PTAX)/PTAX`): zero divergências.
- **Mundo Aberto**: a Ponte aplica 1,50% nas 23 operações; a tabela diz 1,20% ("spread negociado em jul/26"). Kiwi, Canguru e Ponte Aérea (spread em branco ou "padrão" na tabela) têm mediana de 1,50%, o que sugere que "padrão" = 1,5%, mas é **hipótese**. Nórdica (1,25%), Lisboa (1,6%) e EduAbroad (1,3%) batem com o contrato.

### Perguntas que mudam número (para o financeiro)
1. Quem são os lojistas dos ECs 1038867654, 1043300981, 1043396662 e 1076801732 (Atlas), e qual o contrato do Atlas?
2. Rota Global: a taxa +0,50 p.p. desde 12/09 é erro da Orbe ou condição nova? A diferença vira devolução ao lojista?
3. Vigência (Bridge): vale a data do movimento ou da venda original, inclusive nos estornos?
4. Estornos de vendas de agosto: a taxa devolvida reduz a comissão de setembro?
5. Spread "padrão" é 1,5%? Mundo Aberto vale 1,2% (contrato) ou 1,5% (Ponte)?
6. Operações "com ressalva" e a duplicada `PC-0926-0162` entram na comissão e na nota? Quem absorve o spread negativo?
7. A saída de R$ 300.000 de 30/09 na Aurora está em trânsito? Por que a Aurora fica negativa?

## 10. Prévia v2: comissão pela taxa contratada
Regras aplicadas (respostas do @manuel.freitas no canal `#tarvo-financeiro`):
- **Checkout** é faturado por `valor bruto × taxa da planilha`, não pelo valor cobrado pela Orbe. A diferença entre os dois vira divergência.
- **Estorno** abate a comissão do mês do estorno, usando a taxa válida na **data da venda original**.
- **Atlas (EC 1076801732)** não é faturada até confirmação do comercial.
- **Arredondamento comum** (meio para cima), por operação.
- **Remessa**: comissão = 50% do spread realizado. Spread negativo fica fora da comissão e da nota. Tomadora única: Ponte Câmbio.

Premissas minhas, ainda não confirmadas: antecipação entra na comissão com a taxa da planilha (2,10%); diferença de até R$ 0,01 é arredondamento e não conta como divergência (a Orbe trunca, o contrato arredonda); EC sem mapeamento confirmado fica fora da comissão, em grupo separado.

In [35]:
def parse_vig(s):
    s = str(s or "").strip()
    datas = re.findall(r"\d{2}/\d{2}/\d{4}", s)
    if not datas:
        return pd.Timestamp("1900-01-01"), pd.Timestamp("2999-12-31")
    dt = pd.to_datetime(datas[0], format="%d/%m/%Y")
    if s.lower().startswith("até"):
        return pd.Timestamp("1900-01-01"), dt
    return dt, pd.Timestamp("2999-12-31")

cols_v2 = {"debito": "debito", "credito_a_vista": "cred_vista", "credito_2_6x": "cred_2_6x",
           "credito_7_12x": "cred_7_12x", "pix": "pix", "antecipacao": "antecipacao"}
linhas_t = []
for _, r in tx[tx.ec.notna()].iterrows():
    ini, fim = parse_vig(r.vigencia)
    reg = {"ec": str(r.ec), "lojista": r.lojista, "ini": ini, "fim": fim}
    reg.update({c: pct(r[c]) for c in cols_v2.values()})
    linhas_t.append(reg)
tabela_v2 = pd.DataFrame(linhas_t)
print("Linhas da tabela com EC e vigência:")
display(tabela_v2)

Linhas da tabela com EC e vigência:


,ec,lojista,ini,fim,debito,cred_vista,cred_2_6x,cred_7_12x,pix,antecipacao
0,1075624862,Escola Bridge de Idiomas Ltda,1900-01-01,2026-09-14,1.49,2.59,3.19,3.79,0.89,2.1
1,1075624862,Bridge Idiomas (NOVA),2026-09-15,2999-12-31,1.49,2.19,3.19,3.79,0.89,2.1
2,1074504011,Rota Global Intercâmbio Ltda,2026-01-01,2999-12-31,1.49,2.49,2.99,3.59,0.89,2.1
3,1086291869,Vela,2026-01-01,2999-12-31,1.49,2.69,3.19,3.79,0.89,2.1
4,1058961395,Andes Expedições Ltda,2026-01-01,2999-12-31,1.39,2.69,3.19,3.79,0.89,2.1
5,1018325654,Casa do Estudante Viagens Ltda,2026-01-01,2999-12-31,1.49,2.69,3.19,3.79,0.89,2.1
6,1066100982,Fluente Idiomas Online Ltda,2026-01-01,2999-12-31,1.49,2.69,3.19,3.79,0.79,2.1
7,1037647039,Mundo Aberto,2026-01-01,2999-12-31,1.49,2.49,2.99,3.79,0.89,2.1
8,1020660828,EduAbroad,2026-01-01,2999-12-31,1.49,2.69,3.19,3.49,0.89,2.1
9,1014537966,Dublin Hub Intercâmbio Ltda,2026-06-01,2999-12-31,1.49,2.59,3.19,3.79,0.89,2.1


In [36]:
def taxa_contratada(ec, faixa_, data):
    if faixa_ not in cols_v2 or str(ec) not in set(tabela_v2.ec):
        return None
    c = tabela_v2[(tabela_v2.ec == str(ec)) & (tabela_v2.ini <= data) & (data <= tabela_v2.fim)]
    if c.empty:
        return None
    val = c.iloc[0][cols_v2[faixa_]]
    return None if pd.isna(val) else val

def comissao(bruto, taxa_pct):
    x = dec(bruto) * Decimal(str(round(taxa_pct, 6))) / Decimal(100)
    sinal = -1 if x < 0 else 1
    return sinal * abs(x).quantize(Decimal("0.01"), ROUND_HALF_UP)

RETIDOS = {"1076801732"}   # Atlas Idiomas: não faturar até o comercial confirmar

v2 = d.copy()
v2["data_ref"] = pd.to_datetime(v2.data_venda_original.where(v2.data_venda_original.ne(""), v2.data_movimento),
                                format="%d/%m/%Y")
v2["taxa_contrato"] = [taxa_contratada(e, f, dt) for e, f, dt in zip(v2.cod_estabelecimento, v2.faixa, v2.data_ref)]
v2["status_ec"] = v2.cod_estabelecimento.map(
    lambda e: "retido (comercial)" if e in RETIDOS else ("mapeado" if str(e) in set(tabela_v2.ec) else "sem mapeamento"))
v2["comissao_contrato"] = [comissao(b, t) if t is not None and pd.notna(t) else None
                           for b, t in zip(v2.valor_bruto, v2.taxa_contrato)]
v2["valor_taxa_orbe"] = [dec(t) for t in v2.valor_taxa]
v2["dif_brl"] = [float(o - c) if c is not None else None for o, c in zip(v2.valor_taxa_orbe, v2.comissao_contrato)]
v2["ruido_arred"] = v2.dif_brl.map(lambda x: x is not None and abs(x) <= 0.01)
v2["divergencia"] = v2.dif_brl.map(lambda x: x is not None and abs(x) > 0.01)
v2["comissao_f"] = v2.comissao_contrato.map(lambda c: float(c) if c is not None else None)
print("Linhas por status do EC:")
display(v2.status_ec.value_counts().to_frame("linhas"))

Linhas por status do EC:


,linhas
status_ec,
mapeado,1653
sem mapeamento,233
retido (comercial),57


In [37]:
mapeadas = v2[v2.status_ec == "mapeado"]
print("Linhas mapeadas:", len(mapeadas), "| com taxa contratada encontrada:", int(mapeadas.taxa_contrato.notna().sum()))
print("Diferenças só de arredondamento (≤ R$ 0,01):", int(mapeadas.ruido_arred.sum()))
print("Divergências reais (> R$ 0,01):", int(mapeadas.divergencia.sum()),
      "| soma (+ = Orbe cobrou acima do contrato):", round(mapeadas.loc[mapeadas.divergencia, "dif_brl"].sum(), 2))
resumo_v2 = (mapeadas[mapeadas.divergencia]
             .groupby(["cod_estabelecimento", "tipo_lancamento"])
             .agg(linhas=("id_transacao", "size"), dif_brl=("dif_brl", "sum")).round(2))
display(resumo_v2)

Linhas mapeadas: 1653 | com taxa contratada encontrada: 1653
Diferenças só de arredondamento (≤ R$ 0,01): 1561
Divergências reais (> R$ 0,01): 92 | soma (+ = Orbe cobrou acima do contrato): 1575.8


linhas  dif_brl
cod_estabelecimento tipo_lancamento                 
1074504011          ESTORNO               4   -98.42
                    VENDA                83  1644.28
1075624862          VENDA                 5    29.94

In [38]:
# Comissão de checkout por lojista (taxa contratada), separada em MDR e antecipação
mapeadas = v2[v2.status_ec == "mapeado"].copy()
mapeadas["linha"] = mapeadas.tipo_lancamento.map(lambda t: "antecipacao" if "ANTECIPACAO" in t else "MDR")
mapeadas["lojista"] = mapeadas.cod_estabelecimento.map(tabela_v2.groupby("ec").lojista.first())
com_lojista = mapeadas.pivot_table(index="lojista", columns="linha", values="comissao_f",
                                   aggfunc="sum", fill_value=0).round(2)
com_lojista["total"] = com_lojista.sum(axis=1).round(2)
display(com_lojista.sort_values("total", ascending=False))
print("Total checkout (mapeados):", round(com_lojista.total.sum(), 2),
      "| notas: 1 por lojista, total de", len(com_lojista), "lojistas")

linha,MDR,antecipacao,total
lojista,,,
Mundo Aberto,17337.07,1185.02,18522.09
Oxford Prep Cursos Ltda,9529.99,6547.91,16077.90
Rota Global Intercâmbio Ltda,14495.62,1125.65,15621.27
Casa do Estudante Viagens Ltda,12564.35,2827.57,15391.92
Maple Study Assessoria Educacional Ltda,12476.24,2534.54,15010.78
EduAbroad,10312.30,1757.87,12070.17
Escola Bridge de Idiomas Ltda,11916.02,0.00,11916.02
Dublin Hub Intercâmbio Ltda,9390.27,2515.81,11906.08
Canguru,8917.78,2761.65,11679.43


Total checkout (mapeados): 194943.54 | notas: 1 por lojista, total de 18 lojistas


In [39]:
# Remessa: comissão = 50% do spread realizado; spread negativo fora; duplicata contada uma vez
rem = op_u.copy()
rem["comissao_remessa"] = [(Decimal(str(s)) * Decimal("0.5")).quantize(Decimal("0.01"), ROUND_HALF_UP) if s > 0
                           else Decimal("0.00") for s in rem.spread_brl_f]
rem["comissao_f"] = rem.comissao_remessa.map(float)
tab_nome = tx.drop_duplicates("cnpj14").set_index("cnpj14").lojista
rem["lojista"] = rem.cnpj14.map(tab_nome).fillna(rem.cliente)
rem_res = (rem.groupby("lojista")
           .agg(operacoes=("num_op", "size"), negativas=("spread_brl_f", lambda s: int((s <= 0).sum())),
                spread_positivo=("spread_brl_f", lambda s: round(s[s > 0].sum(), 2)),
                comissao=("comissao_f", "sum"))
           .round(2).sort_values("comissao", ascending=False))
display(rem_res)
print("Nota única para a Ponte Câmbio (tomadora):", round(rem.comissao_f.sum(), 2))
print("Fora da comissão e da nota (spread ≤ 0):", int((rem.spread_brl_f <= 0).sum()), "operações")
print("Incluídas mesmo com 'APROVADO C/ RESSALVA' (Manuel: seguem normalmente):", int((rem.status_compliance != "APROVADO").sum()))

,operacoes,negativas,spread_positivo,comissao
lojista,,,,
Rota Global Intercâmbio Ltda,28,0,9836.52,4918.34
Maple Study Assessoria Educacional Ltda,25,0,8589.91,4295.03
Canguru,23,1,7528.01,3764.05
Mundo Aberto,23,0,7095.25,3547.69
Lisboa Cursos e Viagens Ltda,16,0,6392.62,3196.36
Casa do Estudante Viagens Ltda,18,0,6151.74,3075.91
Ponte Aérea Estudantil Ltda,15,1,6014.55,3007.30
Nórdica Educação Internacional Ltda,18,0,5828.25,2914.18
Kiwi Experience,15,0,5336.64,2668.36


Nota única para a Ponte Câmbio (tomadora): 39543.45
Fora da comissão e da nota (spread ≤ 0): 4 operações
Incluídas mesmo com 'APROVADO C/ RESSALVA' (Manuel: seguem normalmente): 16


In [40]:
# Contestação com a Ponte: spread realizado diferente do contratado (mesma tabela da seção 8.2)
print("Contestar com a Ponte (spread realizado ≠ contratado):")
display(pc[pc.situacao == "DIFERENTE"][["cliente_ponte", "pct_mediana", "spread_contratado_pct", "dif_pp", "spread_brl"]])
pag_atlas = au[au.lancamento.str.contains("ATLAS IDIOMAS")]
print("Atlas: PIX pagos pela Aurora:", len(pag_atlas), "| total R$", round(pag_atlas.valor_f.sum(), 2),
      "| sem faturamento (retido). Pagamento sem nota vira pendência.")

Contestar com a Ponte (spread realizado ≠ contratado):


,cliente_ponte,pct_mediana,spread_contratado_pct,dif_pp,spread_brl
cnpj14,,,,,
17217888000111,MUNDO ABERTO INTERCÂ,1.5,1.2,0.3,7095.25


Atlas: PIX pagos pela Aurora: 2 | total R$ -249301.77 | sem faturamento (retido). Pagamento sem nota vira pendência.
